# FerroSim — Direct Waveform — Version 4

**Legendre Try-compatible optimization, statistical reporting, and texture-evolution analysis.** This is the next notebook version; v3 and earlier files are retained. No synthetic measured rewards are used.

## Scientific goal

Optimize time-dependent applied fields on the same 20×20 FerroSim physical lattice, using the same measured terminal absolute-curl reward. The 900 physical field samples are optimized directly, rather than coefficients or grammar operations, while maintaining the same experiment configuration, physical oracle, statistics, per-disorder comparisons and texture-state archive.

**Reference:** `Legendre Try/FerroSim_Legendre_Clean_Standalone_Audited.ipynb` (November 24, 2026). The major new result is the progression from best-so-far measured reward to the **actual polarization textures** obtained at each improvement. Texture metrics are diagnostics, not added optimization objectives.

This is a pure **900-coordinate waveform-space genetic algorithm**, not a surrogate-guided search. The three matched arms are **Hard-Crossover-GA** (one- and two-point splice), **Smooth-Crossover-GA** (blended splice), and **Mutation-Only-GA**. All three can use Gaussian-spike, step, and segment-replacement mutations. No amplitude clipping, slope limit, endpoint projection, or automatic waveform repair is imposed.

## Anaconda execution

Create an Anaconda environment with Python 3.11, NumPy, pandas, SciPy, Matplotlib, Numba, JupyterLab/IPython and a platform-compatible PyTorch build. Open this notebook in JupyterLab, start a **fresh kernel**, edit the first Python cell, then run **all cells from the top**. Notebook cells create their own Python modules and results under `FerroSim_Direct_Waveform_v4/`; no GitHub access is needed to execute them.

QUICK and FULL both use RK4 with the **same maximum subdivision budget**, so numerical validity conditions do not change between modes. QUICK and FULL use exactly the same GA operators and physical oracle; generation/seed settings determine the total cost. A low-data software check can run with `N_DISORDERS=1, SEEDS=[0], INITIAL_CONTROLS=2, GENERATIONS=1` as a low-data diagnostic. The default mirrors Legendre Try's 20-map, 64-initial-control, one-seed, one-generation run; it can still be computationally costly.

For both 1 and N disorders, the first map is the original reference and each additional map permutes the positions of the same 19 defect vectors. The number of maps, number of stochastic optimizer seeds, and generation budget remain independent.

**Statistics:** exact evaluation to reward targets, per-map waveforms and Pareto fronts, complexity-retention at 95/99%, source/selection attribution, explicitly non-applicable DKL calibration, paired disorder-block comparisons, optional source–target transfer, and optional higher-fidelity physical validation.

**Texture section:** saved NPZ states for every distinct best-so-far improvement and a CSV timeline referencing unchanged incumbents; T1 physical state snapshots, T2 texture metrics through generations, T3 reward/texture descriptors, T4 physical-state PCA. The objective remains terminal curl only.

**Interpretation caveats:** periodic curl diagnostic with open-edge physical evolution; resolution of the 900-point driving waveform is not independently refined by additional RK4 substeps; no global optimality or physical novelty is certified by code alone. Full benchmark execution and numerical results must be checked on the user's Anaconda environment.

In [ ]:
# ── SINGLE CONFIGURATION CELL; keep all other notebook cells unchanged ──
MODE = "QUICK"                 # "QUICK" or "FULL"; physics stays RK4
N_DISORDERS = 20               # any integer >=1, including 1
SEEDS = [0]                    # nonnegative unique optimizer seeds, e.g. [0,1,2]
GENERATIONS = 1                # 0=initialization only; 32=full study
INITIAL_CONTROLS = 64          # >=2; 64 recommended for scientific comparisons

ANALYZE_EACH_DISORDER = True
RUN_CROSS_DISORDER_TRANSFER = True
RUN_HIGHER_FIDELITY_VALIDATION = True
VALIDATE_ALL_DISORDERS = False
RUN_TEXTURE_ANALYSIS = True
TEXTURE_PLOT_EACH_DISORDER = True
TEXTURE_MAX_SNAPSHOTS = 6
TEXTURE_PCA_REFERENCE_ONLY = True

from pathlib import Path
import os,sys,math,json,hashlib,subprocess,base64,zlib
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
assert MODE in ("QUICK","FULL")
assert isinstance(N_DISORDERS,(int,np.integer)) and N_DISORDERS>=1
assert isinstance(INITIAL_CONTROLS,(int,np.integer)) and INITIAL_CONTROLS>=2
assert isinstance(GENERATIONS,(int,np.integer)) and GENERATIONS>=0
assert SEEDS and len(set(SEEDS))==len(SEEDS) and all(isinstance(s,(int,np.integer)) and s>=0 for s in SEEDS)
assert isinstance(TEXTURE_MAX_SNAPSHOTS,int) and TEXTURE_MAX_SNAPSHOTS>=1
REPRESENTATION="Direct_Waveform"
PROJECT=Path.cwd()/"FerroSim_Direct_Waveform_v4"
LIB=PROJECT/"lib"
for p in (PROJECT,LIB):p.mkdir(parents=True,exist_ok=True)
print("Representation:",REPRESENTATION,"| runtime:",PROJECT.resolve())

## 1. Embedded physical model, optimizer and analytic modules

The embedded numerical oracle is the same FerroSim physics used in the five v4 notebooks. The Legendre Try advanced-analysis code is harmonized through a small adapter for the different representations. Every source cell is editable and writes a local module using Jupyter's `%%writefile` magic.

In [ ]:
%%writefile FerroSim_Direct_Waveform_v4/lib/representation.py
"""Direct 900-sample E(t)/Ec genotype; no amplitude/slew/endpoint constraints.

Waveform-level genetic operators: Gaussian spikes, step mutations, segment
replacement, one- and two-point hard crossover, and blended crossover.
"""
import numpy as np
from scipy.signal import find_peaks
DIM=DRIVE_SAMPLES=900

def fields(x,n=900):
    x=np.asarray(x,float)
    single=x.ndim==1
    if single:x=x[None,:]
    if x.ndim!=2 or x.shape[1]!=900:raise ValueError("Requires 900 native samples")
    if int(n)<2:raise ValueError("n must be >=2")
    if int(n)==900:out=x
    else:
        t=np.linspace(0,1,900);u=np.linspace(0,1,int(n))
        out=np.array([np.interp(u,t,row) for row in x])
    return out[0] if single else out

def complexity(x,n=4097):
    w=np.atleast_2d(fields(x,n))
    z={key:np.zeros(len(w),int) for key in ("turns","raw_turns","turns_1pct","turns_5pct")}
    for i,row in enumerate(w):
        span=float(np.ptp(row))
        if span>0 and np.isfinite(span):
            for key,p in [("raw_turns",0),("turns_1pct",.01),("turns",.02),("turns_5pct",.05)]:
                z[key][i]=len(find_peaks(row,prominence=p*span)[0])+len(find_peaks(-row,prominence=p*span)[0])
    z["peak_ec"]=np.max(np.abs(w),axis=1)
    z["rms_ec"]=np.sqrt(np.mean(w*w,axis=1))
    return z

def _amplitude(rng,parent=None):
    if parent is None:scale=np.exp(rng.normal(np.log(25.),1.))
    else:scale=max(1.,float(np.sqrt(np.mean(np.asarray(parent)**2))))*np.exp(rng.normal(0.,1.))
    return float(rng.choice([-1.,1.])*scale)

def initial_design(seed,n=64):
    if not isinstance(n,(int,np.integer)) or n<2:raise ValueError("Need >=2 initial waveforms")
    rng=np.random.default_rng(int(seed));t=np.linspace(0,1,900)
    gauss=lambda center,width:np.exp(-.5*((t-center)/width)**2)
    waves=[np.zeros(900),30*gauss(.5,.11),-30*gauss(.35,.07),
           np.where((t>=.2)&(t<.7),20.,-10.),
           40*t-20.,45*np.sin(6*np.pi*t),
           np.interp(t,[0,.13,.29,.53,.76,1],[0,25,-30,0,40,-12]),
           np.where(t<.5,35.,-35.),
           25*np.sin(4*np.pi*t)+16*gauss(.76,.04)]
    while len(waves)<n:
        w=np.zeros(900)
        for _ in range(int(rng.integers(1,6))):
            op=int(rng.integers(4));amp=_amplitude(rng)
            if op==0:
                w+=amp*gauss(rng.uniform(0,1),np.exp(rng.uniform(np.log(.003),np.log(.22))))
            elif op==1:
                a,b=sorted(rng.choice(900,2,replace=False));w[int(a):int(b)+1]+=amp
            elif op==2:
                k=int(rng.integers(2,9))
                knots=np.r_[0.,np.sort(rng.uniform(0,1,k-1)),1.]
                w+=np.interp(t,knots,rng.normal(0,abs(amp),size=k+1))
            else:w+=amp*np.sin(2*np.pi*rng.integers(1,17)*t+rng.uniform(0,2*np.pi))
        waves.append(w)
    return np.asarray(waves[:n],float)

def gaussian_spike(parent,rng):
    child=np.array(parent,float,copy=True)
    center=rng.uniform(0,899);width=np.exp(rng.uniform(np.log(1.),np.log(250.)))
    child+=_amplitude(rng,parent)*np.exp(-.5*((np.arange(900)-center)/width)**2)
    return child,"gaussian_spike"

def step_mutation(parent,rng):
    child=np.array(parent,float,copy=True)
    a,b=sorted(rng.choice(900,2,replace=False))
    child[int(a):int(b)+1]+=_amplitude(rng,parent)
    return child,"step_mutation"

def segment_replacement(parent,rng):
    child=np.array(parent,float,copy=True)
    a,b=sorted(rng.choice(900,2,replace=False));a=int(a);b=int(b)+1
    u=np.linspace(0,1,b-a);amp=_amplitude(rng,parent)
    choice=int(rng.integers(4))
    if choice==0:piece=np.full(b-a,amp)
    elif choice==1:piece=np.linspace(amp,-amp,b-a)
    elif choice==2:piece=amp*np.sin(2*np.pi*rng.integers(1,7)*u+rng.uniform(0,2*np.pi))
    else:piece=amp*np.exp(-.5*((u-rng.uniform())/rng.uniform(.02,.35))**2)
    child[a:b]=piece
    return child,"segment_replacement"

def mutate(parent,rng):
    return (gaussian_spike,step_mutation,segment_replacement)[int(rng.integers(3))](parent,rng)

def one_point_crossover(a,b,rng):
    k=int(rng.integers(1,900))
    return np.r_[a[:k],b[k:]],"one_point_crossover"

def two_point_crossover(a,b,rng):
    l,r=sorted(rng.choice(np.arange(1,900),2,replace=False))
    child=np.array(a,copy=True);child[int(l):int(r)]=b[int(l):int(r)]
    return child,"two_point_crossover"

def smooth_crossover(a,b,rng):
    k=int(rng.integers(1,900));w=int(rng.integers(2,180))
    l=max(0,k-w//2);r=min(899,k+w//2)
    child=np.array(a,copy=True)
    alpha=np.linspace(0.,1.,r-l+1)
    child[l:r+1]=(1-alpha)*a[l:r+1]+alpha*b[l:r+1]
    child[r+1:]=b[r+1:]
    return child,"smooth_crossover"

def proposed_child(method,a,b,rng):
    if method=="Mutation-Only-GA":return mutate(a,rng)
    if rng.random()<.7:
        if method=="Hard-Crossover-GA":
            result,op=(one_point_crossover if rng.random()<.5 else two_point_crossover)(a,b,rng)
        elif method=="Smooth-Crossover-GA":result,op=smooth_crossover(a,b,rng)
        else:raise ValueError("Unknown GA variant")
        if rng.random()<.3:
            result,mutation=mutate(result,rng);op=op+"+"+mutation
        return result,op
    return mutate(a,rng)


In [ ]:
%%writefile FerroSim_Direct_Waveform_v4/lib/oracle.py
"""Accelerated deterministic FerroSim GLD evaluator.

Physics: native rama_dev_updated@5eb258e, gamma=1, tetragonal, k=100,
dep_alpha=.3, finite-grid interactions with fixed /4 neighbor normalization.
Default integration is subdivided RK4, NOT the native first-order Euler method.
No field-amplitude bound or state clipping is imposed in RK4. The native
calDeriv clipping guard |P|=10 is explicitly diagnosed if ever crossed.
Native periodic central-difference curl and an interior diagnostic are returned.
"""
from __future__ import annotations
import os
import math
from pathlib import Path
import numpy as np
os.environ.setdefault("NUMBA_NUM_THREADS", "2")
from numba import njit, prange

N = 20
ALPHA1, ALPHA2, ALPHA3 = -1.6, 12.2, 40.0
COUPLING, DEP_ALPHA = 100.0, 0.3
EC = (-2.0 / 3.0) * ALPHA1 * np.sqrt(abs(-ALPHA1/(3.0*ALPHA2)))
TIME = np.linspace(0.0, 4.0, 1200)[:950]
DT = float(TIME[1]-TIME[0])
SOURCE_COMMIT = "5eb258e2a47d10b3d9bdb9e94dd9879b8410024c"
_LOCAL_DEFECT_PATH = Path(os.environ.get("FERROSIM_DEFECT_PATH", str(Path(__file__).with_name("defect_list_seed.npy"))))
DEFAULT_DEFECT_PATH = (_LOCAL_DEFECT_PATH if _LOCAL_DEFECT_PATH.exists()
                       else Path(__file__).resolve().parents[1]/"GA-DKL/defect_list_seed.npy")
_DEFECTS = None


def default_defects():
    global _DEFECTS
    if _DEFECTS is None:
        _DEFECTS = np.load(DEFAULT_DEFECT_PATH, allow_pickle=False).astype(np.float64).reshape(N,N,2)
    return _DEFECTS.copy()


@njit(cache=True)
def _rhs(p, ex, defects, coupling, dep_alpha, native_clip):
    n = p.shape[1]
    out = np.empty_like(p)
    mx, my = 0.0, 0.0
    for i in range(n):
        for j in range(n):
            mx += p[0,i,j]
            my += p[1,i,j]
    mx /= n*n
    my /= n*n
    for i in range(n):
        for j in range(n):
            x, y = p[0,i,j], p[1,i,j]
            if native_clip:
                x = min(10.,max(-10.,x))
                y = min(10.,max(-10.,y))
            sx, sy = 0.0, 0.0
            # Same finite-grid graph as native knn_search(r=1.1).
            if i>0:
                sx += p[0,i-1,j]; sy += p[1,i-1,j]
            if j>0:
                sx += p[0,i,j-1]; sy += p[1,i,j-1]
            if j<n-1:
                sx += p[0,i,j+1]; sy += p[1,i,j+1]
            if i<n-1:
                sx += p[0,i+1,j]; sy += p[1,i+1,j]
            out[0,i,j] = -(2*ALPHA1*x + 4*ALPHA2*x*x*x + 2*ALPHA3*y*y*x
                           + coupling*(x-sx/4.) - ex + dep_alpha*mx - defects[i,j,0])
            out[1,i,j] = -(2*ALPHA1*y + 4*ALPHA2*y*y*y + 2*ALPHA3*x*x*y
                           + coupling*(y-sy/4.) + dep_alpha*my - defects[i,j,1])
    return out


@njit(cache=True)
def _maxabs(p):
    q = 0.0
    for k in range(p.size):
        v = abs(p.flat[k])
        if not np.isfinite(v):
            return np.inf
        if v>q:
            q=v
    return q


@njit(cache=True)
def _curl(p):
    n=p.shape[1]
    c=np.empty((n,n),dtype=np.float64)
    total, interior = 0.,0.
    for i in range(n):
        for j in range(n):
            v=.5*(p[1,(i+1)%n,j]-p[1,(i-1)%n,j]-p[0,i,(j+1)%n]+p[0,i,(j-1)%n])
            c[i,j]=v
            total+=abs(v)
            if 0<i<n-1 and 0<j<n-1:
                interior+=abs(v)
    return total,interior,c


@njit(cache=True)
def _integrate(field, defects, initial, fidelity, method, history, coupling, dep_alpha, max_substeps):
    """method 0: native-style Euler, method 1: curvature-subdivided RK4."""
    p=initial.copy()
    steps=field.size
    n=p.shape[1]
    hst=np.full((2,steps,n,n),np.nan,dtype=np.float64) if history else np.empty((2,1,n,n),dtype=np.float64)
    hst[:,0,:,:]=p
    peak=_maxabs(p)
    defects_peak=0.
    for q in range(defects.size):
        defects_peak=max(defects_peak,abs(defects.flat[q]))
    rhs_calls=0
    subdivisions=0
    guard_exceeded=peak>10.
    status=0
    for t in range(1,steps):
        e0,e1=field[t-1]*EC,field[t]*EC
        if method==0:
            count=fidelity
        else:
            # A conservative estimate of the local Jacobian row-sum magnitude.
            # The forcing equilibrium scale anticipates sharp field increases.
            pbound=max(_maxabs(p),((max(abs(e0),abs(e1))+defects_peak+10.)/(4*ALPHA2))**(1/3)+.3)
            curvature=(12*ALPHA2+6*abs(ALPHA3))*pbound*pbound+2*coupling+abs(2*ALPHA1)+dep_alpha
            needed=DT*curvature*fidelity
            # Check in floating point before integer conversion: finite but huge
            # requested fields can otherwise overflow the subdivision counter.
            if not np.isfinite(needed) or needed>max_substeps:
                status=2
                break
            count=max(2*fidelity,int(math.ceil(needed)))
        if count>max_substeps:
            status=2
            break
        h=DT/count
        for sub in range(count):
            f0=sub/count
            f1=(sub+1)/count
            if method==0:
                # At fidelity=1 this exactly uses field[t], as native calcODE does.
                ex=e0+(e1-e0)*f1
                p=p+h*_rhs(p,ex,defects,coupling,dep_alpha,True)
                rhs_calls+=1
            else:
                ea=e0+(e1-e0)*f0
                eb=e0+(e1-e0)*f1
                em=.5*(ea+eb)
                k1=_rhs(p,ea,defects,coupling,dep_alpha,False)
                k2=_rhs(p+.5*h*k1,em,defects,coupling,dep_alpha,False)
                k3=_rhs(p+.5*h*k2,em,defects,coupling,dep_alpha,False)
                k4=_rhs(p+h*k3,eb,defects,coupling,dep_alpha,False)
                p=p+(h/6.)*(k1+2*k2+2*k3+k4)
                rhs_calls+=4
            pm=_maxabs(p)
            if not np.isfinite(pm):
                status=1
                break
            peak=max(peak,pm)
            guard_exceeded=guard_exceeded or pm>10.
        subdivisions+=count
        if history:
            hst[:,t,:,:]=p
        if status:
            break
    if not history:
        hst[:,0,:,:]=p
    if status:
        return np.nan,np.nan,p,peak,guard_exceeded,status,rhs_calls,subdivisions,hst
    reward,interior,_=_curl(p)
    return reward,interior,p,peak,guard_exceeded,status,rhs_calls,subdivisions,hst


@njit(cache=True)
def _batch(fields,defects,initial,fidelity,method,coupling,dep_alpha,max_substeps):
    batch=fields.shape[0]
    n=initial.shape[1]
    reward=np.empty(batch); interior=np.empty(batch); peak=np.empty(batch)
    final=np.empty((batch,2,n,n)); guards=np.empty(batch,dtype=np.bool_)
    status=np.empty(batch,dtype=np.int64); rhs=np.empty(batch,dtype=np.int64); subdivisions=np.empty(batch,dtype=np.int64)
    for q in range(batch):
        r,j,p,m,g,s,nrhs,nsub,_=_integrate(fields[q],defects,initial,fidelity,method,False,coupling,dep_alpha,max_substeps)
        reward[q]=r; interior[q]=j; final[q]=p; peak[q]=m
        guards[q]=g; status[q]=s; rhs[q]=nrhs; subdivisions[q]=nsub
    return reward,interior,final,peak,guards,status,rhs,subdivisions


def evaluate_fields(fields_ec, fidelity=1, return_history=False, method="rk4", defects_ec=None,
                    initial_p=None, coupling=COUPLING, dep_alpha=DEP_ALPHA, max_substeps=100000):
    """Evaluate x-only waveforms with a fresh state for every candidate.

    Parameters
    ----------
    fields_ec : (batch,900) or (batch,950), or a single corresponding vector.
        Actual native-interface field E/Ec, WITHOUT normalization or amplitude cap.
        With 900 samples, 50 repeats of the final field are appended.
    fidelity : integer >=1
        Multiplies RK4 temporal resolution. Legacy Euler uses this many substeps.
    return_history : bool
        Add P_history shape (batch,2,950,n,n), mean_P and curl_history.
    method : "rk4" (default) or "legacy_euler"
        RK4 uses linear field interpolation and no state clipping.
    initial_p : optional (n,n,2) or (2,n,n), default zeros on20x20.
    defects_ec : optional (n*n,2) or (n,n,2), default published19-site array.

    status=0 success; 1 nonfinite integration; 2 subdivision resource limit.
    native_guard_exceeded flags |P|>10 without clipping or discarding the result.
    """
    arr=np.asarray(fields_ec,dtype=np.float64)
    if arr.ndim==1:
        arr=arr[None,:]
    if arr.ndim!=2 or arr.shape[1] not in (900,950):
        raise ValueError("Expected (batch,900) or (batch,950) native E/Ec values")
    if not np.isfinite(arr).all():
        raise ValueError("All supplied fields must be finite")
    if arr.shape[1]==900:
        arr=np.concatenate((arr,np.repeat(arr[:,-1:],50,axis=1)),axis=1)
    arr=np.ascontiguousarray(arr)
    if int(fidelity)!=fidelity or fidelity<1:
        raise ValueError("fidelity must be an integer >=1")
    if int(max_substeps)!=max_substeps or max_substeps<1:
        raise ValueError("max_substeps must be an integer >=1")
    if method not in ("rk4","legacy_euler"):
        raise ValueError("Unknown integration method")
    method_id=int(method=="rk4")
    if initial_p is None:
        initial=np.zeros((2,N,N),dtype=np.float64)
    else:
        initial=np.asarray(initial_p,dtype=np.float64)
        if initial.shape[-1]==2 and initial.shape[0]!=2:
            initial=initial.transpose(2,0,1)
        if initial.ndim!=3 or initial.shape[0]!=2 or initial.shape[1]!=initial.shape[2]:
            raise ValueError("Invalid initial polarization shape")
        initial=np.ascontiguousarray(initial)
    n=initial.shape[1]
    defects=default_defects() if defects_ec is None else np.asarray(defects_ec,dtype=np.float64).reshape(n,n,2)
    if defects.shape!=(n,n,2):
        raise ValueError("Defect grid must match initial polarization grid")
    if not np.isfinite(initial).all() or not np.isfinite(defects).all():
        raise ValueError("Initial polarization and defects must be finite")
    defects=np.ascontiguousarray(defects*EC)
    if return_history:
        out=[_integrate(x,defects,initial,int(fidelity),method_id,True,float(coupling),float(dep_alpha),int(max_substeps)) for x in arr]
        r,j,p,m,g,s,nrhs,nsub,hist=map(np.asarray,zip(*out))
    else:
        r,j,p,m,g,s,nrhs,nsub=_batch(arr,defects,initial,int(fidelity),method_id,float(coupling),float(dep_alpha),int(max_substeps))
    result={"reward":r,"curl":r,"interior":j,"final_P":p,"max_abs_P":m,
            "native_guard_exceeded":g,"status":s,"valid":(s==0)&np.isfinite(r),
            "rhs_calls":nrhs,"substeps":nsub,"field_peak_ec":np.max(np.abs(arr),axis=1),
            "fidelity":int(fidelity),"method":method}
    if return_history:
        result["P_history"]=hist
        result["mean_P"]=hist.mean(axis=(3,4))
        ch=np.empty((arr.shape[0],arr.shape[1]))
        ih=np.empty_like(ch)
        for b in range(arr.shape[0]):
            for t in range(arr.shape[1]):
                ch[b,t],ih[b,t],_=_curl(hist[b,:,t])
        result["curl_history"]=ch
        result["interior_history"]=ih
        result["time"]=TIME.copy()
        result["field_ec"]=arr
    return result


def curl_field(p):
    """Native periodic centered curl diagnostic for P shaped (2,n,n)."""
    return _curl(np.asarray(p,dtype=np.float64))[2]


In [ ]:
%%writefile FerroSim_Direct_Waveform_v4/lib/experiment.py
"""Pure GA benchmark operating directly on 900-point physical field waveforms.

Three methods have the same starting population and exactly the same measured
physical evaluation budget. No DKL surrogate and no waveform repair/clipping.
A source/config-fingerprinted directory and atomic checkpoint allow restart.
"""
import argparse
import json
import os
import pickle
import tempfile
from pathlib import Path
import numpy as np
import pandas as pd
from representation import initial_design,fields,complexity,proposed_child
from oracle import evaluate_fields

METHODS=["Hard-Crossover-GA","Smooth-Crossover-GA","Mutation-Only-GA"]

def _atomic_pickle(path,state):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    fd,tmp=tempfile.mkstemp(dir=path.parent,prefix="checkpoint_")
    try:
        with os.fdopen(fd,"wb") as h:
            pickle.dump(state,h,pickle.HIGHEST_PROTOCOL)
            h.flush();os.fsync(h.fileno())
        os.replace(tmp,path)
    finally:
        if os.path.exists(tmp):os.unlink(tmp)

def _atomic_npz(path,**arrays):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    fd,tmp=tempfile.mkstemp(dir=path.parent,prefix="arrays_")
    try:
        with os.fdopen(fd,"wb") as h:
            np.savez_compressed(h,**arrays)
            h.flush();os.fsync(h.fileno())
        os.replace(tmp,path)
    finally:
        if os.path.exists(tmp):os.unlink(tmp)

def measure(waves,defects):
    x=np.atleast_2d(np.asarray(waves,float))
    if x.ndim!=2 or x.shape[1]!=900 or not np.isfinite(x).all():
        raise ValueError("Physical evaluation requires finite 900-sample waveforms")
    result=evaluate_fields(fields(x,900),method="rk4",fidelity=1,
                           defects_ec=defects,max_substeps=100000)
    valid=np.asarray(result["valid"],bool)&~np.asarray(result["native_guard_exceeded"],bool)
    return dict(reward=np.where(valid,result["reward"],np.nan),valid=valid,
        interior=np.asarray(result["interior"],float),
        status=np.asarray(result["status"],int),
        rhs_calls=np.asarray(result["rhs_calls"],int),
        substeps=np.asarray(result["substeps"],int))

def initial_cache(out,mapname,seed,defects,count,run_id):
    path=Path(out)/mapname/f"shared_initial_seed_{seed}.npz"
    if path.is_file():
        with np.load(path,allow_pickle=False) as data:
            if data["run_id"].item()!=run_id:raise ValueError("Incompatible initial cache")
            wave=data["waveforms"].copy()
            res={key:data[key].copy() for key in ("reward","valid","interior","status","rhs_calls","substeps")}
        if len(wave)!=count:raise ValueError("Initial-cache size mismatch")
        return wave,res
    waves=initial_design(seed,count)
    result=measure(waves,defects)
    _atomic_npz(path,run_id=np.asarray(run_id),waveforms=waves,**result)
    return waves,result

def records(waves,result,mapname,method,seed,generation,offset,ops,selection,parents):
    descriptors=complexity(waves)
    out=[]
    for i in range(len(waves)):
        pa,pb=parents[i]
        out.append({
            "map":str(mapname),"method":str(method),"seed":int(seed),
            "evaluation":offset+i+1,"generation":int(generation),
            "reward":float(result["reward"][i]),"valid":bool(result["valid"][i]),
            "interior":float(result["interior"][i]),"status":int(result["status"][i]),
            "rhs_calls":int(result["rhs_calls"][i]),"substeps":int(result["substeps"][i]),
            "turns":int(descriptors["turns"][i]),"complexity":int(descriptors["turns"][i]),
            "raw_turns":int(descriptors["raw_turns"][i]),
            "peak_ec":float(descriptors["peak_ec"][i]),
            "rms_ec":float(descriptors["rms_ec"][i]),
            "operation":str(ops[i]),"proposal_source":str(ops[i]),
            "selection_source":str(selection[i]),
            "parent_evaluation_a":int(pa),"parent_evaluation_b":int(pb),
            "predicted_mean":np.nan,"predicted_std":np.nan})
    return out

def tournament(waves,rewards,rng):
    eligible=np.flatnonzero(np.isfinite(rewards))
    if not len(eligible):raise RuntimeError("No physically valid parents available for GA")
    chosen=rng.choice(eligible,size=3,replace=len(eligible)<3)
    index=int(chosen[np.argmax(rewards[chosen])])
    return waves[index],index+1

def _export(folder,rows,waves):
    if len(rows)!=len(waves):raise RuntimeError("Mismatch of waveforms and physical observations")
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    df=pd.DataFrame(rows)
    df.to_csv(folder/"history.csv",index=False)
    _atomic_npz(folder/"waveforms.npz",waveforms=np.asarray(waves,float))
    return df

def run_one(out,mapname,defects,method,seed,initial_count,generations,batch,run_id):
    folder=Path(out)/mapname/method/f"seed_{seed}"
    folder.mkdir(parents=True,exist_ok=True)
    checkpoint=folder/"checkpoint.pkl"
    rng_seed=(int(seed)*1000003+1009*sum(map(ord,method))
              +97*sum(map(ord,mapname)))%(2**63-1)
    rng=np.random.default_rng(rng_seed)
    if checkpoint.is_file():
        # Trusted LOCAL pickle only; never accept an untrusted external checkpoint.
        with checkpoint.open("rb") as h:state=pickle.load(h)
        if state["run_id"]!=run_id or state["seed"]!=seed or state["method"]!=method:
            raise ValueError("Checkpoint source/configuration mismatch")
        rows=state["rows"];waves=state["waves"]
        done=int(state["completed_generation"])
        rng.bit_generator.state=state["rng_state"]
        if len(rows)!=initial_count+done*batch:raise RuntimeError("Corrupt checkpoint history")
    else:
        waves,initial=initial_cache(out,mapname,seed,defects,initial_count,run_id)
        rows=records(waves,initial,mapname,method,seed,0,0,
                     ["initial_design"]*len(waves),["shared_initial"]*len(waves),
                     [(0,0)]*len(waves))
        done=0
        _atomic_pickle(checkpoint,dict(run_id=run_id,seed=seed,method=method,
            completed_generation=done,rows=rows,waves=waves,
            rng_state=rng.bit_generator.state))
    if done>generations:raise ValueError("Checkpoint extends beyond requested run")
    for generation in range(done+1,generations+1):
        reward=np.asarray([r["reward"] for r in rows],float)
        new=[];ops=[];parent_ids=[]
        for _ in range(batch):
            a,aid=tournament(waves,reward,rng)
            b,bid=tournament(waves,reward,rng)
            child,op=proposed_child(method,a,b,rng)
            new.append(child);ops.append(op)
            parent_ids.append((aid,bid if "crossover" in op else 0))
        new=np.asarray(new,float)
        measured=measure(new,defects)
        rows.extend(records(new,measured,mapname,method,seed,generation,len(rows),
                            ops,["genetic_search"]*batch,parent_ids))
        waves=np.vstack([waves,new])
        _atomic_pickle(checkpoint,dict(run_id=run_id,seed=seed,method=method,
            completed_generation=generation,rows=rows,waves=waves,
            rng_state=rng.bit_generator.state))
        print(f"{mapname}/{method}/seed{seed}: generation={generation},"
              f" valid={int(measured['valid'].sum())}/{batch}",flush=True)
    return _export(folder,rows,waves)

def run(out,map_dir,mapnames,seeds,initial_count,generations,batch,run_id):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    results=[]
    for mapname in mapnames:
        defects=np.load(Path(map_dir)/(mapname+".npy"),allow_pickle=False)
        for seed in seeds:
            for method in METHODS:
                print("Direct waveform GA:",mapname,seed,method,flush=True)
                results.append(run_one(out,mapname,defects,method,int(seed),
                    initial_count,generations,batch,run_id))
    df=pd.concat(results,ignore_index=True)
    planned=len(mapnames)*len(seeds)*len(METHODS)*(initial_count+generations*batch)
    if len(df)!=planned:raise RuntimeError(f"Incomplete physical history {len(df)} != {planned}")
    df.to_csv(out/"all_evaluations.csv",index=False)
    print("Recorded method-history observations:",len(df),
          "valid:",int(df["valid"].sum()),flush=True)

if __name__=="__main__":
    p=argparse.ArgumentParser()
    p.add_argument("--out",required=True)
    p.add_argument("--map-dir",required=True)
    p.add_argument("--map-names",required=True)
    p.add_argument("--seeds",required=True)
    p.add_argument("--initial",type=int,default=64)
    p.add_argument("--generations",type=int,default=32)
    p.add_argument("--batch",type=int,default=32)
    p.add_argument("--run-id",required=True)
    a=p.parse_args()
    if a.initial<2 or a.generations<0 or a.batch<1:p.error("Need initial>=2, generations>=0, batch>=1")
    seeds=[int(s) for s in a.seeds.split(",")]
    if not seeds or min(seeds)<0 or len(seeds)!=len(set(seeds)):
        p.error("Seeds must be unique nonnegative integers")
    maps=a.map_names.split(",")
    run(a.out,a.map_dir,maps,seeds,a.initial,a.generations,a.batch,a.run_id)


In [ ]:
%%writefile FerroSim_Direct_Waveform_v4/lib/common_analysis.py
"""FerroSim v4 common analysis and representation adapters.
Embedded in each standalone notebook. The Legendre Try statistics remain the baseline.
No surrogate predictions are represented as measured physics.
"""
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from representation import fields, complexity

METHODS=[]
NAMES={}

def configure_methods(representation):
    global METHODS,NAMES
    if representation in ("Fourier","Legendre","Spline"):
        METHODS[:]=["ga_dkl","gflownet_dkl","diffusion_dkl","qd_dkl"]
        NAMES.clear()
        NAMES.update({"ga_dkl":"GA–DKL","gflownet_dkl":"GFlowNet–DKL",
                      "diffusion_dkl":"Diffusion–DKL","qd_dkl":"QD–DKL"})
    elif representation=="Segments":
        METHODS[:]=["GA-DKL","QD-DKL","Sequential-GFlowNet-inspired-DKL",
                   "Denoising-diffusion-inspired-DKL"]
        NAMES.clear();NAMES.update({m:m for m in METHODS})
    elif representation=="Grammar":
        METHODS[:]=["simple/GA-DKL","simple/QD-DKL","full/GA-DKL","full/QD-DKL"]
        NAMES.clear();NAMES.update({m:m.replace("/"," / ") for m in METHODS})
    elif representation=="Direct_Waveform":
        METHODS[:]=["Hard-Crossover-GA","Smooth-Crossover-GA","Mutation-Only-GA"]
        NAMES.clear();NAMES.update({m:m.replace("-"," ") for m in METHODS})
    else:raise ValueError("Unknown representation "+str(representation))
    return METHODS,NAMES

def load_histories(root,representation):
    root=Path(root)
    configure_methods(representation)
    if representation in ("Fourier","Legendre","Spline"):
        from analysis import load
        df=load(root)
        if df.empty:raise RuntimeError("No physical histories in "+str(root))
    else:
        path=root/"all_evaluations.csv"
        if not path.is_file():raise FileNotFoundError(path)
        df=pd.read_csv(path)
        if df.empty:raise RuntimeError("No recorded evaluations")
        df["native_method"]=df["method"].astype(str)
        if representation=="Grammar":
            df["method"]=df["grammar"].astype(str)+"/"+df["native_method"]
        df["complexity"]=df["turns"].astype(int)
        if "raw_turns" not in df:df["raw_turns"]=np.nan
        if "selection_source" not in df:df["selection_source"]="dkl_acquisition"
        if "proposal_source" not in df:
            src=("operator" if "operator" in df else "operation")
            df["proposal_source"]=df[src].fillna("unknown").astype(str)
        if "substeps" not in df:df["substeps"]=0
        if "rhs_calls" not in df:df["rhs_calls"]=0
        vectors=[None]*len(df)
        cols=["map","seed","method"] + (["grammar"] if representation=="Grammar" else [])
        for key,g in df.groupby(cols,sort=False):
            row=g.iloc[0]
            mid=str(row["map"]);seed=int(row["seed"]);native=str(row["native_method"])
            if representation=="Segments":
                file=root/mid/native/("seed_"+str(seed))/"coordinates.npz"
                key_name="x"
            elif representation=="Direct_Waveform":
                file=root/mid/native/("seed_"+str(seed))/"waveforms.npz"
                key_name="waveforms"
            else:
                file=root/mid/(str(row["grammar"])+"__"+native)/("seed_"+str(seed))/"programs.npz"
                key_name="programs"
            with np.load(file,allow_pickle=False) as archive:arr=archive[key_name].copy()
            order=g.sort_values("evaluation")
            if len(arr)!=len(order):
                raise RuntimeError("Coordinate archive length disagrees with measured history: "+str(file))
            for idx,vector in zip(order.index,arr):vectors[int(idx)]=vector
        if any(v is None for v in vectors):
            raise RuntimeError("Missing serialized controls in historical evaluation rows")
        df["coefficients"]=vectors
    for key,default in (("predicted_mean",np.nan),("predicted_std",np.nan),
                        ("interior",np.nan),("peak_ec",np.nan),("rms_ec",np.nan),
                        ("substeps",0),("rhs_calls",0),("selection_source","not_recorded"),
                        ("proposal_source","not_recorded")):
        if key not in df:df[key]=default
    if df["valid"].dtype==object:
        df["valid"]=df["valid"].astype(str).str.strip().str.lower().eq("true")
    else:df["valid"]=df["valid"].astype(bool)
    df["method"]=df["method"].astype(str)
    if not set(df.method).issubset(set(METHODS)):
        raise ValueError("Unexpected method labels "+str(set(df.method)-set(METHODS)))
    df["evaluation"]=df["evaluation"].astype(int)
    df["generation"]=df["generation"].astype(int)
    df["seed"]=df["seed"].astype(int)
    return df

def valid(df):
    return df[df.valid & np.isfinite(df.reward)].copy()

def pareto(df):
    q=valid(df)
    if q.empty:return q
    rows=[];best=-np.inf
    for comp in sorted(q.complexity.unique()):
        g=q[q.complexity==comp];r=g.loc[g.reward.idxmax()]
        if float(r.reward)>best+1e-12:
            rows.append(r);best=float(r.reward)
    return pd.DataFrame(rows) if rows else q.iloc[:0]

def incumbent_history(df):
    z=df.sort_values("evaluation");best=-np.inf;idx=None;out=[]
    for _,row in z.iterrows():
        if bool(row.valid) and np.isfinite(row.reward) and float(row.reward)>best:
            best=float(row.reward);idx=row
        out.append(idx)
    return z,out

def _save(fig,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    fig.savefig(path,dpi=150,bbox_inches="tight")
    plt.show();plt.close(fig)

def plot_convergence(df,figures,baseline_fn):
    fig,axs=plt.subplots(1,2,figsize=(13,4.7),layout="constrained")
    for method in METHODS:
        g=df[(df.method==method)&(df["map"]=="reference")]
        for j,(seed,sub) in enumerate(g.groupby("seed")):
            sub,history=incumbent_history(sub)
            vals=[float(r.reward) if r is not None else np.nan for r in history]
            axs[0].step(sub.evaluation,vals,where="post",alpha=.6,label=NAMES[method] if j==0 else None)
        traces=[]
        for (mid,seed),sub in df[df.method==method].groupby(["map","seed"]):
            context=df[(df["map"]==mid)&(df.seed==seed)]
            baseline=float(baseline_fn[(mid,seed)])
            observed=context.loc[context.valid & np.isfinite(context.reward),"reward"]
            target=float(observed.max()) if len(observed) else np.nan
            sub,h=incumbent_history(sub)
            reward=np.array([float(r.reward) if r is not None else np.nan for r in h])
            denom=target-baseline
            normalized=(reward-baseline)/denom if denom>1e-12 else np.zeros(len(reward))
            traces.append(normalized)
        if traces:
            n=min(map(len,traces));a=np.asarray([z[:n] for z in traces])
            axs[1].plot(np.arange(1,n+1),np.nanmean(a,axis=0),label=NAMES[method])
            if len(a)>1:
                se=np.nanstd(a,axis=0,ddof=1)/np.sqrt(len(a))
                axs[1].fill_between(np.arange(1,n+1),np.nanmean(a,axis=0)-se,
                     np.nanmean(a,axis=0)+se,alpha=.1)
    axs[0].set(title="Reference disorder: best measured curl",xlabel="Physical evaluations / method",
               ylabel="Best terminal curl")
    axs[1].set(title="Fraction of pooled gain above best starting control",
               xlabel="Physical evaluations / method",ylabel="Normalized observed gain")
    for ax in axs:ax.grid(alpha=.2);ax.legend(fontsize=7)
    _save(fig,Path(figures)/"02_convergence.png")

def plot_reference_pareto(df,figures):
    ref=valid(df[df["map"]=="reference"])
    fig,axs=plt.subplots(1,len(METHODS),figsize=(5*len(METHODS),4.8),squeeze=False,layout="constrained")
    for ax,method in zip(axs.flat,METHODS):
        sub=ref[ref.method==method]
        for generation,g in sub.groupby("generation"):
            ax.scatter(g.complexity,g.reward,s=9,alpha=.3)
        front=pareto(sub)
        if len(front):ax.plot(front.complexity,front.reward,"ko-",lw=1.2,ms=3)
        ax.set(title=NAMES[method],xlabel="Resolved turns (2% prominence)",ylabel="Curl reward")
        ax.grid(alpha=.2)
    _save(fig,Path(figures)/"03_reference_pareto.png")

def plot_control_evolution(df,figures,representation,seed=None):
    """Reference-disorder, single-seed (not pooled) coefficient+waveform evolution."""
    ref=df[df["map"]=="reference"]
    seed=int(ref.seed.min()) if seed is None else int(seed)
    fig,axs=plt.subplots(len(METHODS),2,figsize=(13,3.1*len(METHODS)),layout="constrained")
    for i,m in enumerate(METHODS):
        g=ref[(ref.method==m)&(ref.seed==seed)]
        if g.empty:continue
        waves=[];coordinates=[]
        for gen in sorted(g.generation.unique()):
            upto=valid(g[g.generation<=gen])
            if upto.empty:continue
            w=upto.loc[upto.reward.idxmax()]
            wave=np.asarray(fields(w.coefficients,n=900),float).reshape(-1)
            waves.append(wave)
            if representation in ("Fourier","Legendre","Spline"):
                from representation import full_coefficients
                coordinates.append(np.asarray(full_coefficients(w.coefficients),float).ravel())
            elif representation=="Direct_Waveform":
                coordinates.append(wave-waves[0])
            else:coordinates.append(np.asarray(w.coefficients,float).ravel())
        if not waves:continue
        coords=np.stack(coordinates);wavs=np.stack(waves)
        for ax,arr,label in [(axs[i,0],coords,("Physical basis coefficients" if representation in ("Fourier","Legendre","Spline") else "Change since initial incumbent" if representation=="Direct_Waveform" else "Encoded program/segments")),
                             (axs[i,1],wavs,"Applied waveform E/Ec")]:
            vmax=max(float(np.nanmax(np.abs(arr))),1e-10)
            ax.imshow(arr,aspect="auto",origin="lower",cmap="coolwarm",vmin=-vmax,vmax=vmax)
            ax.set(title=NAMES[m]+" | "+label,xlabel="Index / control time",ylabel="Generation")
    fig.suptitle("Best-so-far evolution on reference map, optimizer seed "+str(seed))
    _save(fig,Path(figures)/"04_control_evolution.png")

def plot_champion_and_descriptor(df,figures,representation):
    ref=valid(df[df["map"]=="reference"])
    if ref.empty:return
    champion=ref.loc[ref.reward.idxmax()]
    simplest=ref[ref.complexity<=1]
    choose=[("Best",champion)]
    if len(simplest):choose.append(("Best <=1 turn",simplest.loc[simplest.reward.idxmax()]))
    fig,axs=plt.subplots(1,2,figsize=(12,4.5),layout="constrained")
    for label,row in choose:
        waveform=np.asarray(fields(row.coefficients,900),float).reshape(-1)
        axs[0].plot(np.linspace(0,1,len(waveform)),waveform,label=label+" R="+format(row.reward,".4g"))
        if representation in ("Fourier","Legendre","Spline"):
            from representation import full_coefficients
            z=np.asarray(full_coefficients(row.coefficients)).ravel()
        else:z=np.asarray(row.coefficients).ravel()
        axs[1].plot(np.arange(len(z)),z,label=label)
    axs[0].set(title="Champion and simple physical controls",xlabel="Normalized time",ylabel="E/Ec")
    axs[1].set(title="Direct waveform genome" if representation=="Direct_Waveform" else "Descriptor/encoded parameters",
               xlabel="Waveform sample" if representation=="Direct_Waveform" else "Descriptor index",
               ylabel="E/Ec" if representation=="Direct_Waveform" else "Parameter value")
    for ax in axs:ax.grid(alpha=.2);ax.legend(fontsize=8)
    _save(fig,Path(figures)/"05_best_waveform_and_descriptor.png")

def plot_modal_energy(df,figures,representation):
    if representation not in ("Fourier","Legendre","Spline"):
        print("Modal energy plot not applicable to variable-topology representation.")
        return
    from representation import full_coefficients
    ref=valid(df[df["map"]=="reference"])
    champion=ref.loc[ref.reward.idxmax()]
    simpler=ref[ref.complexity<=1]
    comparisons=[("Best",champion)]
    if len(simpler):comparisons.append(("Simple",simpler.loc[simpler.reward.idxmax()]))
    fig,axs=plt.subplots(1,2,figsize=(12,4.6),layout="constrained")
    for label,row in comparisons:
        a=np.asarray(full_coefficients(row.coefficients)).ravel()
        axs[0].plot(np.arange(len(a)),a,label=label)
        if representation=="Legendre":
            energy=a*a/(2*np.arange(len(a))+1)
        elif representation=="Fourier":
            energy=a*a/2
        else:
            # Spline basis is not orthogonal; show physically rendered RMS and label.
            w=np.asarray(fields(row.coefficients,900)).ravel()
            energy=np.full(len(a),np.nan)
        if representation!="Spline":axs[1].plot(np.arange(len(energy)),energy,label=label)
    axs[0].set(title="Physical expansion coefficients",xlabel="Mode / basis index",ylabel="Amplitude")
    axs[1].set(title="Orthogonal modal energy" if representation!="Spline" else "Spline modal energy not additive",
               xlabel="Mode index",ylabel="Contribution to mean squared field")
    if representation=="Spline":
        axs[1].text(.5,.5,"Spline basis is nonorthogonal.\nCompare rendered RMS E/Ec.",ha="center",va="center",transform=axs[1].transAxes)
    for ax in axs:ax.grid(alpha=.2)
    axs[0].legend()
    _save(fig,Path(figures)/"05_modal_descriptors.png")

def plot_disorder_summary(df,figures,tables,maps):
    v=valid(df);rows=[]
    for mid in maps:
        g=v[v["map"]==mid]
        if g.empty:continue
        best=g.loc[g.reward.idxmax()]
        simple=g[g.complexity<=1]
        rows.append({"map":mid,"best_reward":best.reward,"best_turns":best.complexity,
         "best_simple_reward":float(simple.reward.max()) if len(simple) else np.nan})
    t=pd.DataFrame(rows);t.to_csv(Path(tables)/"disorder_summary.csv",index=False)
    if len(maps)>1:
        fig,axs=plt.subplots(1,2,figsize=(12,4),layout="constrained")
        axs[0].plot(t["map"],t.best_reward,"o-",label="Best")
        axs[0].plot(t["map"],t.best_simple_reward,"s--",label="Simple")
        axs[1].bar(t["map"],t.best_turns)
        axs[0].set(title="Disorder-dependent reward",ylabel="Curl")
        axs[1].set(title="Winner complexity",ylabel="Resolved turns")
        for ax in axs:ax.tick_params(axis="x",rotation=65);ax.grid(alpha=.2)
        axs[0].legend()
        _save(fig,Path(figures)/"06_disorder_summary.png")
    return t

def plot_all_disorder_pareto(df,maps,figures):
    import math
    columns=min(4,len(maps));rows=int(math.ceil(len(maps)/columns))
    fig,axs=plt.subplots(rows,columns,figsize=(4.3*columns,3.6*rows),squeeze=False,layout="constrained")
    for ax in axs.flat:ax.axis("off")
    for ax,mid in zip(axs.flat,maps):
        g=valid(df[df["map"]==mid])
        for method,z in g.groupby("method"):
            p=pareto(z)
            if len(p):ax.plot(p.complexity,p.reward,marker=".",lw=1.1,label=NAMES.get(method,method))
        pooled=pareto(g)
        if len(pooled):ax.plot(pooled.complexity,pooled.reward,"k--",lw=1.7,label="Pooled")
        ax.axis("on");ax.set(title=mid,xlabel="Resolved turns",ylabel="Curl");ax.grid(alpha=.2)
    if len(maps)<=4:axs.flat[0].legend(fontsize=6)
    _save(fig,Path(figures)/"13_all_disorder_pareto_montage.png")


def write_selected(df,tables):
    """Save physically evaluated winners and low-turn alternatives, for every disorder."""
    tables=Path(tables)
    selected=tables/"selected_controls";selected.mkdir(parents=True,exist_ok=True)
    rows=[]
    for mid,g in valid(df).groupby("map"):
        champion=g.loc[g.reward.idxmax()]
        simple=g[g.complexity<=1]
        picks=[("best",champion)]
        if len(simple):picks.append(("simple",simple.loc[simple.reward.idxmax()]))
        for label,row in picks:
            wave=np.asarray(fields(row.coefficients,n=900),float).reshape(-1)
            fname=("".join(q if q.isalnum() or q in "-_" else "_" for q in str(mid))+"__"+label+".npz")
            np.savez_compressed(selected/fname,
                coordinates=np.asarray(row.coefficients,float),waveform_Ec=wave,
                measured_reward=float(row.reward),generation=int(row.generation),
                evaluation=int(row.evaluation),seed=int(row.seed))
            rows.append(dict(map=mid,kind=label,method=row.method,seed=int(row.seed),
                             evaluation=int(row.evaluation),generation=int(row.generation),
                             reward=float(row.reward),complexity=int(row.complexity),
                             waveform_file=str(Path("selected_controls")/fname)))
    result=pd.DataFrame(rows)
    result.to_csv(tables/"selected_controls.csv",index=False)
    return result


In [ ]:
%%writefile FerroSim_Direct_Waveform_v4/lib/advanced_analysis.py
"""Transparent, offline analyses of observed FerroSim histories.

Everything in this module consumes measured physical outcomes. Figures handle a
single map, a single seed, and zero optimization generations. No synthetic
rewards, extrapolation or fabricated uncertainty intervals are generated.
"""
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from representation import fields, complexity
from common_analysis import METHODS, NAMES, pareto


def _saved(fig, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)


def _valid(df):
    return df.loc[df.valid & np.isfinite(df.reward)].copy()


def _best(g):
    valid = _valid(g)
    return None if valid.empty else valid.loc[valid.reward.idxmax()]


def _safe_name(x):
    return ''.join(c if (c.isalnum() or c in ('-','_')) else '_' for c in str(x))


def audit_budget(df, maps, seeds, initial_count, generations, tables, figures, shared_initial_cache=True):
    tables,figures = Path(tables),Path(figures)
    plan = len(maps)*len(seeds)*len(METHODS)*(initial_count+32*generations)
    distinct = (len(maps)*len(seeds)*(initial_count+len(METHODS)*32*generations)
                if shared_initial_cache else plan)
    rows=[]
    for (mp, seed, method),g in df.groupby(['map','seed','method'],sort=False):
        good = _valid(g)
        rows.append({'map':mp,'seed':seed,'method':method,'attempts':len(g),
            'valid':len(good),'invalid':len(g)-len(good),
            'initialization':int((g.generation==0).sum()),
            'new_queries':int((g.generation>0).sum()),
            'best_curl':float(good.reward.max()) if len(good) else np.nan,
            'best_interior_curl':float(good.interior.max()) if len(good) else np.nan,
            'best_turns':int(good.loc[good.reward.idxmax()].complexity) if len(good) else np.nan,
            'peak_ec_best':float(good.loc[good.reward.idxmax()].peak_ec) if len(good) else np.nan,
            'rhs_calls':int(g.rhs_calls.sum()) if 'rhs_calls' in g else 0,
            'substeps':int(g.substeps.sum()) if 'substeps' in g else 0})
    summary = pd.DataFrame(rows)
    summary.to_csv(tables/'run_audit.csv',index=False)
    status = {'recorded_method_observations':int(len(df)),
        'expected_method_observations':int(plan),
        'expected_distinct_physical_calls':int(distinct),
        'valid_recorded_observations':int(df.valid.sum()),
        'invalid_recorded_observations':int((~df.valid).sum()),
        'shared_initial_cache':bool(shared_initial_cache),
        'n_maps':int(len(maps)),'n_seeds':int(len(seeds)),
        'initial_controls_per_map_seed':int(initial_count),
        'new_evaluations_per_method_generation':32,
        'generations':int(generations)}
    (tables/'budget.json').write_text(json.dumps(status,indent=2))
    if len(df)!=plan: raise AssertionError(f'Incomplete physical histories: {len(df)} / {plan}')
    if len(summary)!=len(maps)*len(seeds)*len(METHODS): raise AssertionError('Missing method/seed/map history')
    if (summary.initialization!=initial_count).any(): raise AssertionError('Missing initial observations')
    if (summary.new_queries!=32*generations).any(): raise AssertionError('Missing new observations')
    print('Evaluation accounting:', status)
    return summary


def shared_initial_baselines(df):
    """Per-map, per-seed best VALID initial control pooled across methods.
    Modal runs share the initial observations; Grammar may not.
    """
    df = _valid(df)
    init=df[df.generation==0]
    # Common baseline across methods; initial control populations may differ for Grammar.
    return init.groupby(['map','seed']).reward.max().to_dict()


def exact_target_attainment(df,tables):
    b=shared_initial_baselines(df)
    records=[]
    for mp,g in df.groupby('map'):
        gv=_valid(g)
        target=float(gv.reward.max()) if len(gv) else np.nan
        for (method,seed),r in g.groupby(['method','seed']):
            baseline=float(b[(mp,seed)])
            for fraction in (.95,.99):
                threshold=baseline+fraction*(target-baseline)
                hit=r[r.valid & (r.reward>=threshold)]
                records.append({'map':mp,'method':method,'seed':int(seed),
                    'baseline_best_initial':baseline,'pooled_observed_target':target,
                    'fraction':fraction,'threshold':threshold,
                    'reached':not hit.empty,
                    'first_evaluation':int(hit.evaluation.min()) if len(hit) else np.nan,
                    'initially_attained':bool(len(hit) and int(hit.evaluation.min())<=int((r.generation==0).sum()))})
    t=pd.DataFrame(records)
    t.to_csv(Path(tables)/'exact_target_attainment.csv',index=False)
    return t


def plot_pareto_by_disorder(df, map_ids, figures, tables, all_maps=True):
    root=Path(figures)/'pareto_by_disorder';root.mkdir(parents=True,exist_ok=True)
    p_rows=[]
    selected=map_ids if all_maps else map_ids[:1]
    for mid in selected:
        g=_valid(df[df['map']==mid]); fig,axs=plt.subplots(1,len(METHODS),figsize=(5*len(METHODS),4.7),squeeze=False,layout='constrained')
        for ax,method in zip(axs.flat,METHODS):
            h=g[g.method==method]
            if not h.empty:
                ax.scatter(h.complexity,h.reward,s=11,alpha=.20)
                p=pareto(h)
                if len(p): ax.plot(p.complexity,p.reward,'o-',ms=3.5,lw=1.4)
                for _,x in p.iterrows():
                    p_rows.append(dict(map=mid,method=method,complexity=int(x.complexity),
                        reward=float(x.reward),seed=int(x.seed),evaluation=int(x.evaluation)))
            ax.set_title(NAMES[method]);ax.set(xlabel='Prominent turns (2%)',ylabel='Measured curl');ax.grid(alpha=.2)
        fig.suptitle('Observed reward–complexity Pareto fronts | '+str(mid))
        _saved(fig,root/(_safe_name(mid)+'.png'))
    pd.DataFrame(p_rows,columns=['map','method','complexity','reward','seed','evaluation']).to_csv(Path(tables)/'pareto_by_disorder.csv',index=False)


def plot_best_waveforms_by_disorder(df, map_ids, figures, tables, all_maps=True):
    root=Path(figures)/'champions_by_disorder';root.mkdir(parents=True,exist_ok=True)
    result=[];selected=map_ids if all_maps else map_ids[:1]
    for mid in selected:
        g=_valid(df[df['map']==mid]);fig,axs=plt.subplots(1,len(METHODS),figsize=(5*len(METHODS),4.7),squeeze=False,layout='constrained')
        for ax,method in zip(axs.flat,METHODS):
            h=g[g.method==method];w=_best(h)
            if w is not None:
                wave=np.asarray(fields(w.coefficients,n=900),float).reshape(-1)
                ax.plot(np.linspace(0,1,len(wave)),wave,lw=1.5,label='Best')
                simple=h[h.complexity<=1];s=_best(simple)
                if s is not None:
                    sw=np.asarray(fields(s.coefficients,n=900),float).reshape(-1)
                    ax.plot(np.linspace(0,1,len(sw)),sw,lw=1.0,ls='--',label='Best ≤1 turn')
                ax.legend(fontsize=8)
                result.append({'map':mid,'method':method,'seed':int(w.seed),
                    'evaluation':int(w.evaluation),'best_curl':float(w.reward),
                    'turns':int(w.complexity),'simple_curl':float(s.reward) if s is not None else np.nan})
            ax.set_title(NAMES[method]);ax.set(xlabel='Time / T',ylabel='E / Ec');ax.grid(alpha=.2)
        fig.suptitle('Per-method champion waveforms | '+str(mid))
        _saved(fig,root/(_safe_name(mid)+'.png'))
    pd.DataFrame(result,columns=['map','method','seed','evaluation','best_curl','turns','simple_curl']).to_csv(
        Path(tables)/'champions_by_disorder.csv',index=False)


def complexity_retention(df, figures, tables):
    """Find minimum complexity retaining 95/99% of gain above initialization."""
    base=shared_initial_baselines(df);rows=[]
    for (mid,seed,method),g in df.groupby(['map','seed','method']):
        v=_valid(g)
        if v.empty:continue
        best=v.loc[v.reward.idxmax()]
        baseline=base[(mid,seed)]
        for turns in (0,1,3):
            q=v[v.complexity<=turns]
            selected=_best(q)
            rows.append({'map':mid,'seed':seed,'method':method,'criterion':f'≤{turns} turns',
               'best_reward':float(selected.reward) if selected is not None else np.nan,
               'unrestricted_reward':float(best.reward),'baseline':float(baseline),
               'resolved_turns':int(selected.complexity) if selected is not None else np.nan})
        for fraction in (.95,.99):
            threshold=baseline+fraction*(float(best.reward)-baseline)
            q=v[v.reward>=threshold]
            selected=q.loc[q.complexity.idxmin()] if len(q) else None
            rows.append({'map':mid,'seed':seed,'method':method,
               'criterion':f'{int(100*fraction)}% gain retained',
               'best_reward':float(selected.reward) if selected is not None else np.nan,
               'unrestricted_reward':float(best.reward),'baseline':float(baseline),
               'resolved_turns':int(selected.complexity) if selected is not None else np.nan})
    t=pd.DataFrame(rows);t.to_csv(Path(tables)/'complexity_retention.csv',index=False)
    subset=t[t.criterion.isin(['95% gain retained','99% gain retained'])]
    fig,axs=plt.subplots(1,2,figsize=(12,4.4),layout='constrained')
    for ax,(crit,h) in zip(axs,subset.groupby('criterion',sort=True)):
        a=[h[h.method==m].resolved_turns.dropna().to_numpy() for m in METHODS]
        a=[arr if len(arr) else np.array([np.nan]) for arr in a]
        ax.boxplot(a,positions=np.arange(1,len(METHODS)+1),showfliers=False)
        ax.set_xticks(range(1,len(METHODS)+1),[NAMES[m] for m in METHODS],rotation=25,ha='right')
        ax.set(title=crit,ylabel='Minimum observed prominent turns');ax.grid(axis='y',alpha=.2)
    _saved(fig,Path(figures)/'complexity_retention.png')
    return t


def seed_disorder_variance(df, figures, tables):
    """Separates variation of best observed per-run rewards over seed and disorder."""
    winners=[]
    for (mid,method,seed),g in df.groupby(['map','method','seed']):
        w=_best(g)
        if w is not None: winners.append({'map':mid,'method':method,'seed':int(seed),
            'best_reward':float(w.reward),'best_turns':int(w.complexity)})
    winners=pd.DataFrame(winners);winners.to_csv(Path(tables)/'run_champions.csv',index=False)
    summary=[]
    for method,g in winners.groupby('method'):
        per_map=g.groupby('map').best_reward
        means=per_map.mean()
        within=per_map.agg(lambda x: float(x.var(ddof=1)) if len(x)>1 else np.nan)
        summary.append({'method':method,'maps':len(means),'seeds_per_map':int(g.seed.nunique()),
            'mean_best_curl':float(g.best_reward.mean()),
            'between_map_variance_of_seed_means':float(means.var(ddof=1)) if len(means)>1 else np.nan,
            'mean_within_map_seed_variance':float(within.mean()) if within.notna().any() else np.nan,
            'single_map_or_seed_warning':bool(len(means)<2 or g.seed.nunique()<2)})
    parts=pd.DataFrame(summary);parts.to_csv(Path(tables)/'seed_disorder_variance.csv',index=False)
    fig,axs=plt.subplots(1,2,figsize=(12,4.4),layout='constrained')
    mids=list(dict.fromkeys(winners['map']))
    for method in METHODS:
        h=winners[winners.method==method]
        means=h.groupby('map').best_reward.mean().reindex(mids)
        axs[0].plot(range(len(mids)),means,'o-',ms=3,label=NAMES[method])
    axs[0].set(xticks=range(len(mids)),xticklabels=mids,title='Best curl per map (averaged over seeds)',ylabel='Curl')
    axs[0].tick_params(axis='x',rotation=60);axs[0].legend(fontsize=8);axs[0].grid(alpha=.2)
    arrays=[winners[winners.method==method].best_reward.to_numpy() for method in METHODS]
    axs[1].boxplot(arrays,showfliers=False)
    axs[1].set(xticks=range(1,len(METHODS)+1),xticklabels=[NAMES[m] for m in METHODS],
        title='Best reward across observed map–seed runs',ylabel='Curl')
    axs[1].tick_params(axis='x',rotation=25);axs[1].grid(alpha=.2)
    _saved(fig,Path(figures)/'seed_disorder_variance.png')
    return winners,parts


def proposal_attribution(df, figures, tables):
    # Archive uses two complementary notions: candidate origin and selection channel.
    h=df.copy()
    if 'selection_source' not in h: h['selection_source']='not_recorded'
    if 'proposal_source' not in h: h['proposal_source']='not_recorded'
    rows=[]
    for (m,stage,source),g in h.groupby(['method','selection_source','proposal_source']):
        v=_valid(g)
        rows.append({'method':m,'selection_source':stage,'proposal_source':source,
            'evaluations':len(g),'valid':len(v),
            'best_reward':float(v.reward.max()) if len(v) else np.nan,
            'mean_reward':float(v.reward.mean()) if len(v) else np.nan})
    t=pd.DataFrame(rows);t.to_csv(Path(tables)/'proposal_attribution.csv',index=False)
    fig,axs=plt.subplots(1,2,figsize=(12,4.5),layout='constrained')
    new=h[h.generation>0]
    if len(new):
        x=new.pivot_table(index='method',columns='proposal_source',values='reward',aggfunc='size',fill_value=0).reindex(METHODS)
        x.plot.bar(stacked=True,ax=axs[0],legend=True)
        axs[0].legend(fontsize=7)
        y=new.pivot_table(index='method',columns='selection_source',values='reward',aggfunc='size',fill_value=0).reindex(METHODS)
        y.plot.bar(stacked=True,ax=axs[1],legend=True)
        axs[1].legend(fontsize=7)
        for ax in axs: ax.tick_params(axis='x',rotation=20)
    else:
        for ax in axs: ax.text(.5,.5,'No optimization generations',ha='center',va='center',transform=ax.transAxes)
    axs[0].set_title('Candidates evaluated by origin');axs[1].set_title('Candidates evaluated by selection channel')
    for ax in axs:ax.set(xlabel='Optimizer',ylabel='Physical evaluations');ax.grid(axis='y',alpha=.2)
    _saved(fig,Path(figures)/'proposal_attribution.png')
    return t


def surrogate_calibration(df,figures,tables):
    """Not applicable to the pure GA arms; no invented surrogate diagnostics."""
    cols=["method","n","mean_error","rmse","mae","inside_1_sigma","inside_2_sigma"]
    t=pd.DataFrame(columns=cols)
    t.to_csv(Path(tables)/"surrogate_calibration.csv",index=False)
    fig,ax=plt.subplots(figsize=(10,2.4),layout="constrained")
    ax.text(.5,.55,"Not applicable: direct-waveform GA uses no DKL surrogate.",
            ha="center",va="center",transform=ax.transAxes)
    ax.text(.5,.25,"All recorded rewards are from the physical FerroSim oracle.",
            ha="center",va="center",transform=ax.transAxes)
    ax.axis("off")
    _saved(fig,Path(figures)/"surrogate_calibration.png")
    return t


def plot_additional_complexity_metrics(df,figures,tables):
    """Raw reversals, amplitude and interior curl guard against metric gaming."""
    records=[]
    for (mp,method,seed),g in df.groupby(['map','method','seed']):
        w=_best(g)
        if w is not None:
            wave=np.asarray(fields(w.coefficients,n=900),dtype=float).reshape(-1)
            d=np.diff(wave)
            records.append({'map':mp,'method':method,'seed':seed,'reward':float(w.reward),
                'interior_reward':float(w.interior),'boundary_contribution':float(w.reward-w.interior),
                'boundary_fraction':float((w.reward-w.interior)/w.reward) if w.reward!=0 else np.nan,
                'peak_ec':float(np.max(np.abs(wave))),'total_variation_ec':float(np.sum(np.abs(d))),
                'max_step_ec':float(np.max(np.abs(d))),
                'turns_2pct':int(w.complexity),
                'raw_turns':int(complexity(w.coefficients)['raw_turns'][0])})
    t=pd.DataFrame(records);t.to_csv(Path(tables)/'physical_complexity_metrics.csv',index=False)
    fig,axs=plt.subplots(1,3,figsize=(15,4.0),layout='constrained')
    for method,g in t.groupby('method'):
        name=NAMES.get(method,method)
        axs[0].scatter(g.turns_2pct,g.reward,label=name,s=18,alpha=.65)
        axs[1].scatter(g.total_variation_ec,g.reward,label=name,s=18,alpha=.65)
        axs[2].scatter(g.boundary_fraction,g.reward,label=name,s=18,alpha=.65)
    for ax,xlabel in zip(axs,['Prominent turns','Total variation (E / Ec)','Boundary fraction of curl']):
        ax.set(xlabel=xlabel,ylabel='Best reward per run');ax.grid(alpha=.2)
    axs[0].legend(fontsize=7)
    _saved(fig,Path(figures)/'physical_complexity_metrics.png')
    return t


def export_index(figures,tables,root):
    figs=sorted(str(p.relative_to(root)) for p in Path(figures).rglob('*.png'))
    tabs=sorted(str(p.relative_to(root)) for p in Path(tables).rglob('*') if p.is_file())
    (Path(root)/'output_manifest.json').write_text(json.dumps({'figures':figs,'tables':tabs},indent=2))
    print('Saved',len(figs),'figures and',len(tabs),'tables. Output index:',Path(root)/'output_manifest.json')


def paired_method_comparison(winners, figures, tables, bootstrap_draws=1000):
    """Six paired contrasts with disorder-block bootstrap CIs.
    Seeds are repeated runs on the same disorder, *not* independent disorder
    replicates. Report a confidence interval only for >=2 distinct disorder maps.
    """
    from itertools import combinations
    t=winners.pivot_table(index=['map','seed'],columns='method',values='best_reward',aggfunc='first')
    rows=[];rng=np.random.default_rng(20261124)
    for a,b in combinations(METHODS,2):
        if a not in t or b not in t:continue
        pair=t[[a,b]].dropna()
        if pair.empty:continue
        diff=(pair[a]-pair[b]).astype(float)
        # Within each disorder, average across the available optimizer seeds.
        by_map=diff.groupby(level="map").mean()
        independent_maps=len(by_map)
        if independent_maps>1:
            map_means=by_map.to_numpy()
            ids=rng.integers(independent_maps,size=(bootstrap_draws,independent_maps))
            simulated=map_means[ids].mean(axis=1)
            low,high=np.quantile(simulated,[.025,.975])
        else:low=high=np.nan
        rows.append({'method_a':a,'method_b':b,'paired_map_seed_runs':len(diff),
            'independent_disorder_maps':independent_maps,
            'mean_difference_a_minus_b':float(by_map.mean()),
            'median_difference':float(diff.median()),
            'a_wins':int((diff>0).sum()),'ties':int((diff==0).sum()),
            'b_wins':int((diff<0).sum()),'bootstrap_CI95_low':float(low),
            'bootstrap_CI95_high':float(high),
            'CI_valid':bool(independent_maps>1),
            'interpretation':'Descriptive paired differences, block-bootstrap across disorder maps; no global optimum claim'})
    tab=pd.DataFrame(rows)
    tab.to_csv(Path(tables)/'paired_algorithm_differences.csv',index=False)
    M=np.zeros((len(METHODS),len(METHODS)),dtype=float)
    for _,z in tab.iterrows():
        i=METHODS.index(z.method_a);j=METHODS.index(z.method_b)
        M[i,j]=z.mean_difference_a_minus_b;M[j,i]=-M[i,j]
    fig,ax=plt.subplots(figsize=(8,6),layout='constrained')
    scale=max(float(np.max(abs(M))),1e-8)
    im=ax.imshow(M,cmap='coolwarm',vmin=-scale,vmax=scale)
    ax.set_xticks(range(len(METHODS)),[NAMES[m] for m in METHODS],rotation=35,ha='right')
    ax.set_yticks(range(len(METHODS)),[NAMES[m] for m in METHODS])
    ax.set_title('Paired map-level mean differences: row method minus column method')
    for i in range(len(METHODS)):
        for j in range(len(METHODS)):
            ax.text(j,i,f'{M[i,j]:+.4f}',ha='center',va='center',fontsize=8)
    fig.colorbar(im,ax=ax,shrink=.75,label='Difference in terminal curl')
    _saved(fig,Path(figures)/'paired_algorithm_differences.png')
    return tab


In [ ]:
%%writefile FerroSim_Direct_Waveform_v4/lib/texture_analysis.py
"""FerroSim v4 texture evolution from optimizer incumbents.
For every disorder x method x seed, save the terminal Px, Py and curl whenever
the best-so-far *measured* reward improves. There is no change to the reward.
Identical module is embedded in all five standalone notebooks.
"""
from __future__ import annotations
from pathlib import Path
import math
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import fftconvolve
from oracle import evaluate_fields,curl_field
from representation import fields

def safe_slug(s):
    return re.sub(r"[^0-9a-zA-Z_-]+","_",str(s)).strip("_")

def radial_correlation_length(P):
    """Mean-subtracted, overlap-corrected spatial polarization correlation.
    Reports first 1/e crossing in lattice-pixel units; NaN if unresolved.
    No periodic wraparound is introduced.
    """
    x=np.asarray(P,dtype=float)
    q=x-x.mean(axis=(1,2),keepdims=True)
    variance=float(np.mean(np.sum(q*q,axis=0)))
    if variance<1e-12:return np.nan
    corr=np.zeros((2*x.shape[1]-1,2*x.shape[2]-1),float)
    for component in q:
        corr+=fftconvolve(component,component[::-1,::-1],mode="full")
    ones=np.ones(x.shape[1:])
    overlap=fftconvolve(ones,ones[::-1,::-1],mode="full")
    c=(corr/np.maximum(overlap,1.))/variance
    ny,nx=x.shape[1:]
    iy,ix=np.indices(c.shape)
    rr=np.hypot(iy-(ny-1),ix-(nx-1))
    prev=1.
    for r in range(1,max(2,min(nx,ny)//2+1)):
        ring=(rr>=r-.5)&(rr<r+.5)
        value=float(np.nanmean(c[ring]))
        if not np.isfinite(value):continue
        if value<=math.exp(-1):
            if prev<=math.exp(-1):return float(r)
            return float((r-1)+(prev-math.exp(-1))/max(prev-value,1e-12))
        prev=value
    return np.nan

def state_metrics(P, C, threshold):
    P=np.asarray(P,float);absC=np.abs(np.asarray(C,float))
    total=float(absC.sum())
    squares=float((absC*absC).sum())
    n=absC.size
    localization=float(total*total/(n*squares)) if squares>0 else np.nan
    high=float(np.mean(absC>threshold)) if np.isfinite(threshold) else np.nan
    return {"curl_total":total,"curl_localization":localization,
            "high_curl_area":high,"high_curl_threshold":float(threshold),
            "polarization_rms":float(np.sqrt(np.mean(np.sum(P*P,axis=0)))),
            "polarization_spatial_variance":float(np.mean(np.sum((P-P.mean(axis=(1,2),keepdims=True))**2,axis=0))),
            "correlation_length_px":radial_correlation_length(P)}

def _physical_state(row,defects,max_substeps=100000):
    w=np.asarray(fields(np.asarray(row.coefficients,float),n=900),float).reshape(-1)
    result=evaluate_fields(w,method="rk4",defects_ec=defects,max_substeps=max_substeps)
    good=bool(result["valid"][0]) and not bool(result["native_guard_exceeded"][0])
    if not good:
        raise RuntimeError(f"Previously valid winner failed independent physics reevaluation "
                           f"map={row['map']} method={row.method} seed={row.seed} eval={row.evaluation}")
    P=np.asarray(result["final_P"][0],float)
    C=np.asarray(curl_field(P),float)
    measured=float(result["reward"][0])
    return P,C,w,measured,float(result["interior"][0])

def record_texture_histories(df,map_ids,map_dir,out,*,max_substeps=100000):
    """Build index of every generation and save each distinct incumbent once.
    Returns (generation-level rows, distinct saved states) as DataFrames.
    """
    out=Path(out);archive=out/"states";archive.mkdir(parents=True,exist_ok=True)
    maps=set(map_ids);map_dir=Path(map_dir)
    history=df[df["map"].isin(maps)].copy()
    if history.empty:raise RuntimeError("No optimizer histories available for texture analysis")
    thresholds={}
    baseline_rows=history[(history.generation==0)&history.valid&np.isfinite(history.reward)]
    for (mid,seed),group in baseline_rows.groupby(["map","seed"]):
        best=group.loc[group.reward.idxmax()]
        P,C,w,reward,interior=_physical_state(best,np.load(map_dir/(str(mid)+".npy")),max_substeps)
        threshold=float(np.percentile(np.abs(C),90.))
        thresholds[(str(mid),int(seed))]=threshold
    saved=[];timeline=[];cache={}
    groups=history.groupby(["map","method","seed"],sort=False)
    for (mid,method,seed),group in groups:
        mid=str(mid);method=str(method);seed=int(seed)
        group=group.sort_values("evaluation")
        defects=np.load(map_dir/(mid+".npy"),allow_pickle=False)
        threshold=thresholds.get((mid,seed),np.nan)
        previous=None;previous_c=None;best_reward=-np.inf;winner=None
        for generation,block in group.groupby("generation",sort=True):
            generation=int(generation)
            candidates=block[block.valid & np.isfinite(block.reward)]
            if len(candidates):
                candidate=candidates.loc[candidates.reward.idxmax()]
                if float(candidate.reward)>best_reward:
                    best_reward=float(candidate.reward);winner=candidate
            if winner is None:
                timeline.append({"map":mid,"method":method,"seed":seed,"generation":generation,
                                 "source_evaluation":np.nan,"reward":np.nan,"state_path":None,
                                 "new_record":False})
                continue
            evaluation=int(winner.evaluation)
            key=(mid,method,seed,evaluation)
            is_new=key not in cache
            if is_new:
                P,C,w,physical_reward,interior=_physical_state(winner,defects,max_substeps)
                if not np.isclose(physical_reward,float(winner.reward),rtol=1e-7,atol=1e-8):
                    raise RuntimeError(f"Replay reward disagrees with archived reward for {key}: "
                                       f"{physical_reward} vs {winner.reward}; check oracle and checkpoints")
                state_path=Path("states")/safe_slug(mid)/safe_slug(method)/f"seed_{seed}_eval_{evaluation:06d}.npz"
                target=out/state_path;target.parent.mkdir(parents=True,exist_ok=True)
                np.savez_compressed(target,final_P=P,Px=P[0],Py=P[1],curl=C,abs_curl=np.abs(C),
                                    waveform_Ec=w,reward=np.float64(physical_reward),
                                    interior_curl=np.float64(interior),
                                    generation=np.int64(generation),
                                    evaluation=np.int64(evaluation))
                metrics=state_metrics(P,C,threshold)
                dist=float(np.sqrt(np.mean((P-previous)**2))/
                           max(float(np.sqrt(np.mean(previous**2))),1e-12)) if previous is not None else np.nan
                curl_change=float(np.mean(np.abs(np.abs(C)-np.abs(previous_c)))) if previous_c is not None else np.nan
                saved.append({"map":mid,"method":method,"seed":seed,"generation_first":generation,
                     "evaluation":evaluation,"reward":float(physical_reward),
                     "interior_curl":float(interior),"boundary_curl":float(physical_reward-interior),
                     "state_path":str(state_path),"state_change_from_previous":dist,
                     "curl_change_from_previous":curl_change,**metrics})
                cache[key]={"path":str(state_path),"metrics":metrics,
                            "distance":dist,"curl_change":curl_change}
                previous=P;previous_c=C
            info=cache[key]
            timeline.append({"map":mid,"method":method,"seed":seed,"generation":generation,
                 "source_evaluation":evaluation,"reward":float(best_reward),
                 "state_path":info["path"],"new_record":is_new,
                 "state_change_from_previous":info["distance"] if is_new else 0.,
                 "curl_change_from_previous":info["curl_change"] if is_new else 0.,
                 **info["metrics"]})
    T=pd.DataFrame(timeline);S=pd.DataFrame(saved)
    T.to_csv(out/"texture_generation_index.csv",index=False)
    S.to_csv(out/"texture_distinct_states.csv",index=False)
    (out/"texture_protocol.json").write_text(json.dumps({
       "reward":"terminal sum(abs(periodic centered curl))",
       "what_is_saved":"terminal polarization and curl per unique best-so-far incumbent; unchanged incumbents reference the previous NPZ",
       "timeline":"optimization generation, not within-waveform physical time",
       "curl_localization":"(sum(abs(C))**2)/(pixels*sum(abs(C)**2))",
       "high_curl_threshold":"90th percentile of abs curl of best VALID initial control, across all methods for the same map and seed",
       "correlation_length":"overlap-normalized mean-subtracted vector autocorrelation 1/e crossing; open spatial shifts",
       "p_change":"RMS(Pnew-Pold)/RMS(Pold) for distinct improved records",
       "physical_validation":"independent RK4 re-evaluation with reward agreement check",
       "fields_stored":["Px","Py","curl","abs_curl","waveform_Ec","reward","interior_curl"]
       },indent=2))
    return T,S

def _read(out,record):
    with np.load(Path(out)/str(record.state_path),allow_pickle=False) as z:
        return {"P":z["final_P"].copy(),"C":z["curl"].copy(),"W":z["waveform_Ec"].copy()}

def figure_T1(out,S,*,max_snapshots=6,all_disorders=True):
    """For each (map,method,seed): snapshots of E, Px, Py, |curl|, delta |curl|."""
    out=Path(out);S=S.sort_values(["map","method","seed","generation_first"])
    first=str(S["map"].iloc[0]) if len(S) else "reference"
    count=0
    for (mid,method,seed),g in S.groupby(["map","method","seed"],sort=False):
        if not all_disorders and str(mid)!=first:continue
        inds=np.unique(np.linspace(0,len(g)-1,min(max_snapshots,len(g)),dtype=int))
        chosen=g.iloc[inds]
        stages=[_read(out,row) for _,row in chosen.iterrows()]
        amp=max(max(np.max(np.abs(z["P"])) for z in stages),1e-8)
        curlmax=max(max(np.max(np.abs(z["C"])) for z in stages),1e-8)
        rows=len(stages)
        fig,axs=plt.subplots(rows,5,figsize=(16,2.75*rows),squeeze=False,layout="constrained")
        last=None
        for j,((_,row),state) in enumerate(zip(chosen.iterrows(),stages)):
            P=state["P"];absC=np.abs(state["C"]);w=state["W"]
            axs[j,0].plot(np.linspace(0,1,len(w)),w,color="#007F83",lw=1.2)
            axs[j,0].set(title=f"Gen {int(row.generation_first)}, R={row.reward:.3g}" if j==0 else
                               f"Gen {int(row.generation_first)}, R={row.reward:.3g}",
                         xlabel="t/T",ylabel="E/Ec")
            axs[j,1].imshow(P[0],vmin=-amp,vmax=amp,cmap="coolwarm",origin="lower")
            axs[j,2].imshow(P[1],vmin=-amp,vmax=amp,cmap="coolwarm",origin="lower")
            axs[j,3].imshow(absC,vmin=0,vmax=curlmax,cmap="magma",origin="lower")
            delta=absC-last if last is not None else np.zeros_like(absC)
            dmax=max(curlmax*.5,float(np.max(np.abs(delta))),1e-8)
            axs[j,4].imshow(delta,vmin=-dmax,vmax=dmax,cmap="coolwarm",origin="lower")
            last=absC
        for ax,title in zip(axs[0],["Winning field","P_x","P_y","|curl P|","Change in |curl|"]):
            ax.set_title(title)
        fig.suptitle(f"T1 — Measured physical states as reward improves | {mid} | {method} | seed {seed}")
        p=out/"figures"/"T1"
        p.mkdir(parents=True,exist_ok=True)
        fig.savefig(p/f"T1_{safe_slug(mid)}_{safe_slug(method)}_seed{seed}.png",dpi=135,bbox_inches="tight")
        plt.show();plt.close(fig);count+=1
    print("T1 figures:",count)

def figure_T2(out,T,*,all_disorders=True):
    out=Path(out);figdir=out/"figures"/"T2";figdir.mkdir(parents=True,exist_ok=True)
    for (mid,seed),g in T.groupby(["map","seed"],sort=False):
        if not all_disorders and mid!="reference":continue
        fig,axs=plt.subplots(2,3,figsize=(14,8),layout="constrained")
        fields=[
            ("reward","Best measured curl"),("curl_localization","Curl localization (participation fraction)"),
            ("high_curl_area","Area above fixed initial high-curl threshold"),
            ("state_change_from_previous","Relative P change at new improvements"),
            ("correlation_length_px","P correlation length / pixels"),
            ("curl_change_from_previous","Mean change in |curl| at improvements")]
        for method,h in g.groupby("method"):
            h=h.sort_values("generation")
            for ax,(column,title) in zip(axs.flat,fields):
                ax.plot(h.generation,h[column],marker=".",lw=1.2,label=method)
        for ax,(column,title) in zip(axs.flat,fields):
            ax.set(xlabel="Optimization generation",ylabel=title,title=title)
            ax.grid(alpha=.2)
        axs[0,0].legend(fontsize=7)
        fig.suptitle(f"T2 — Texture descriptors through optimization | {mid} | seed {seed}")
        fig.savefig(figdir/f"T2_{safe_slug(mid)}_seed{seed}.png",dpi=140,bbox_inches="tight")
        plt.show();plt.close(fig)

def figure_T3(out,S):
    out=Path(out)
    props=[("curl_localization","Curl localization"),("high_curl_area","High-curl area"),
           ("state_change_from_previous","Relative change in P"),("correlation_length_px","Correlation length")]
    fig,axs=plt.subplots(2,2,figsize=(12,9),layout="constrained")
    for method,g in S.groupby("method"):
        for ax,(key,label) in zip(axs.flat,props):
            ax.scatter(g[key],g.reward,s=15,alpha=.45,label=method)
    for ax,(key,label) in zip(axs.flat,props):
        ax.set(xlabel=label,ylabel="Best-measured reward",
               title="Measured reward vs "+label)
        ax.grid(alpha=.2)
    axs[0,0].legend(fontsize=7)
    fig.suptitle("T3 — Texture descriptors and reward (repeated incumbents excluded)")
    fig.savefig(out/"figures"/"T3_reward_texture_descriptors.png",dpi=150,bbox_inches="tight")
    plt.show();plt.close(fig)

def figure_T4(out,S,*,reference_only=True):
    out=Path(out)
    maps=["reference"] if reference_only else sorted(S["map"].unique())
    from matplotlib.markers import MarkerStyle
    markers=["o","s","^","D","v","P","X","*"]
    records=[]
    for mid in maps:
        subset=S[S["map"]==mid].copy()
        if len(subset)<2:continue
        vectors=[]
        for _,row in subset.iterrows():
            q=_read(out,row)
            vectors.append(q["P"].reshape(-1))
        X=np.asarray(vectors,float)
        X=X-X.mean(axis=0,keepdims=True)
        if np.linalg.norm(X)<1e-12:
            print("T4 PCA skipped: identical states on",mid);continue
        U,s,Vt=np.linalg.svd(X,full_matrices=False)
        coords=U[:,:2]*s[:2]
        if coords.shape[1]==1:coords=np.c_[coords,np.zeros(len(coords))]
        frac=(s[:2]**2)/(s**2).sum()
        methods=list(dict.fromkeys(subset.method))
        fig,ax=plt.subplots(figsize=(9,6),layout="constrained")
        for j,m in enumerate(methods):
            rows=np.flatnonzero((subset.method==m).to_numpy())
            sc=ax.scatter(coords[rows,0],coords[rows,1],
                c=subset.iloc[rows].reward.to_numpy(float),cmap="viridis",
                marker=markers[j%len(markers)],s=32,alpha=.85,label=m)
            for seed,g in subset[subset.method==m].groupby("seed"):
                ids=[subset.index.get_loc(v) for v in g.sort_values("generation_first").index]
                if len(ids)>1:ax.plot(coords[ids,0],coords[ids,1],alpha=.3,lw=1,color="gray")
        fig.colorbar(sc,ax=ax,label="Measured terminal curl")
        ax.set(xlabel=f"PC1 ({100*frac[0]:.1f}% variance)",
               ylabel=f"PC2 ({100*frac[1]:.1f}% variance)" if len(frac)>1 else "PC2",
               title=f"T4 — Polarization-state PCA | disorder {mid}")
        ax.grid(alpha=.2);ax.legend(fontsize=7)
        fig.savefig(out/"figures"/f"T4_texture_state_PCA_{safe_slug(mid)}.png",dpi=150,bbox_inches="tight")
        plt.show();plt.close(fig)
        for (_,row),(x,y) in zip(subset.iterrows(),coords):
            records.append({"map":mid,"method":row.method,"seed":row.seed,"generation":row.generation_first,
                "evaluation":row.evaluation,"reward":row.reward,"PC1":x,"PC2":y,
                "variance_fraction_PC1":frac[0],"variance_fraction_PC2":frac[1] if len(frac)>1 else 0.})
    pd.DataFrame(records).to_csv(out/"texture_PCA_coordinates.csv",index=False)

def analyze_textures(df,maps,map_dir,root,*,max_substeps=100000,
                     max_snapshots=6,plot_each_disorder=True,plot_pca_reference_only=True):
    root=Path(root);root.mkdir(parents=True,exist_ok=True)
    timeline,saved=record_texture_histories(df,maps,map_dir,root,max_substeps=max_substeps)
    if saved.empty:
        print("No valid incumbent physical states to plot.")
        return timeline,saved
    (root/"figures").mkdir(parents=True,exist_ok=True)
    figure_T1(root,saved,max_snapshots=max_snapshots,all_disorders=plot_each_disorder)
    figure_T2(root,timeline,all_disorders=plot_each_disorder)
    figure_T3(root,saved)
    figure_T4(root,saved,reference_only=plot_pca_reference_only)
    return timeline,saved


## 2. Historical defect map and representation invariants

All representations use the same original 19-site defect lattice. Both two-control smoke initializations and the 64-control research initialization are checked. Modal control bases enforce field and slope endpoint constraints; variable-topology implementations use their own physical compiler.

In [ ]:
REF_B85="c-oun7wQ`j$;eQ~P_3SlTAW;@Zl$1ZlV+i=qoAIaUsO_*m=~X4l#&V(cT3DEP6dh=XCxM+0{I#y1_n9`Mw&VbwF+bcE{0JsARsI7O?HNjtHWS%PcGN8!ix894#OpUE6&gKjJ-L0{6i|p`+h9lcB8k$XilRI$jDo~(<az&#D=fq!$%=rXIw_p*DwWLGy3~pKKB|n@%uK&eo@W_kJ0oxLcv9uMND&#51#gb(VCv40=%@VpVc*8TsMolj@b0hXZTd}(D~t$-+ia*3B8W-av06YqktT+b~F3hU00n))6XykMtK`gZQA5BY~uGoilo|;)}Yb!Its`FS69yYQe7TMv-I<ocLmp>uF>(8p#pejU*CT$cC`I83Qz$6MT1~F"
reference_bytes=zlib.decompress(base64.b85decode(REF_B85.encode()))
(LIB/"defect_list_seed.npy").write_bytes(reference_bytes)
for name in ("representation","oracle","surrogate","gflownet","diffusion","evolution","dkl","experiment","analysis","common_analysis","advanced_analysis","texture_analysis"):
    sys.modules.pop(name,None)
sys.path.insert(0,str(LIB.resolve()))
from representation import fields,complexity
from oracle import evaluate_fields,default_defects,curl_field
from common_analysis import (configure_methods,load_histories,METHODS,NAMES,pareto,
    plot_convergence,plot_reference_pareto,plot_control_evolution,
    plot_champion_and_descriptor,plot_modal_energy,plot_disorder_summary,
    plot_all_disorder_pareto)
from advanced_analysis import (audit_budget,shared_initial_baselines,exact_target_attainment,
    plot_pareto_by_disorder,plot_best_waveforms_by_disorder,complexity_retention,
    seed_disorder_variance,paired_method_comparison,proposal_attribution,surrogate_calibration,
    plot_additional_complexity_metrics,export_index)
from texture_analysis import analyze_textures
configure_methods(REPRESENTATION)
reference=default_defects()
assert reference.shape==(20,20,2)
assert np.count_nonzero(np.any(reference!=0,axis=2))==19
from representation import initial_design
assert np.asarray(initial_design(0,n=2)).shape[0]==2
assert np.asarray(initial_design(0,n=64)).shape[0]==64
print("Reference defect hash:",hashlib.sha256(reference_bytes).hexdigest())
print("All representation and analysis imports succeeded; methods:",METHODS)

## 3. Deterministic disorder ensemble

`N_DISORDERS` includes the historical reference map plus arbitrary additional spatial rearrangements. The map manifest is authoritative; static 4×5 grids are not assumed. Defect strengths and number are preserved, so this is a placement-disorder study.

In [ ]:
import matplotlib.pyplot as plt
MAPS=["reference"]+[f"d{j:02d}" for j in range(N_DISORDERS-1)]
SOURCE_HASHES={name:hashlib.sha256((LIB/name).read_bytes()).hexdigest() for name in ["representation.py","oracle.py","experiment.py","common_analysis.py","advanced_analysis.py","texture_analysis.py"]}
# Compose the physical experiment identity from all search-relevant settings and source files.
SEARCH_CONFIG=dict(representation=REPRESENTATION,mode=MODE,n_disorders=N_DISORDERS,
    seeds=SEEDS,generations=GENERATIONS,initial_count=INITIAL_CONTROLS,
    physical_oracle="rk4",max_substeps=100000,
    source_sha256=SOURCE_HASHES,reference_sha256=hashlib.sha256(reference_bytes).hexdigest())
# Keep different configurations and versions isolated, including modifications to embedded source.
RUN_ID=hashlib.sha256(json.dumps(SEARCH_CONFIG,sort_keys=True).encode()).hexdigest()[:16]
RUN_ROOT=PROJECT/"experiments"/RUN_ID
MAP_DIR=RUN_ROOT/"maps";HISTORY_DIR=RUN_ROOT/"histories"
FIG_DIR=RUN_ROOT/"figures";TABLE_DIR=RUN_ROOT/"tables"
for p in (MAP_DIR,HISTORY_DIR,FIG_DIR,TABLE_DIR):p.mkdir(parents=True,exist_ok=True)
active=reference.reshape(-1,2)[np.any(reference.reshape(-1,2)!=0,axis=1)]
manifest=[]
for i,mid in enumerate(MAPS):
    if i==0:arr=reference.copy();seed=None
    else:
        seed=20261122+i-1
        rg=np.random.default_rng(seed);locations=rg.choice(400,len(active),replace=False)
        arr=np.zeros((400,2));arr[locations]=active[rg.permutation(len(active))]
        arr=arr.reshape(20,20,2)
    np.save(MAP_DIR/(mid+".npy"),arr)
    manifest.append({"id":mid,"seed":seed,"active_sites":int(np.any(arr!=0,axis=2).sum())})
(MAP_DIR/"manifest.json").write_text(json.dumps(manifest,indent=2))
def defect_for(mid):return np.load(MAP_DIR/(str(mid)+".npy"),allow_pickle=False)
ncols=min(5,N_DISORDERS);nrows=int(math.ceil(N_DISORDERS/ncols))
fig,axs=plt.subplots(nrows,ncols,figsize=(2.7*ncols,2.5*nrows),squeeze=False,layout="constrained")
for ax in axs.flat:ax.axis("off")
for ax,mid in zip(axs.flat,MAPS):
    ax.imshow(np.linalg.norm(defect_for(mid),axis=2),origin="lower")
    ax.set_title(mid);ax.axis("on")
fig.savefig(FIG_DIR/"01_disorder_maps.png",dpi=140);plt.show();plt.close(fig)
print("Disorder maps:",len(MAPS),"| experiment:",RUN_ID)

## 4. Three pure waveform-space GAs — matched physical evaluation budgets

The same 900-sample starting population and initial physical observations are shared by all three variants. Hard crossover uses one- or two-point splicing; smooth crossover blends a splice; mutation-only uses no crossover. Every arm can introduce Gaussian spikes, field steps, and segment replacements. No waveform amplitude/slope/continuity constraints are imposed. The physical reward remains terminal absolute curl. There is no DKL surrogate.

In [ ]:
# Three pure GA variants. No DKL, amplitude clipping or waveform repair.
hyper={"methods":["Hard-Crossover-GA","Smooth-Crossover-GA","Mutation-Only-GA"],
       "mutation_operators":["gaussian_spike","step_mutation","segment_replacement"],
       "crossovers":["one_point","two_point","smooth"],
       "offspring_per_generation":32,"uses_surrogate":False}
cmd=[sys.executable,str(LIB/"experiment.py"),
     "--out",str(HISTORY_DIR.resolve()),
     "--map-dir",str(MAP_DIR.resolve()),
     "--map-names",",".join(MAPS),"--seeds",",".join(map(str,SEEDS)),
     "--initial",str(INITIAL_CONTROLS),"--generations",str(GENERATIONS),
     "--batch","32","--run-id",RUN_ID]
env=os.environ.copy()
env.update({"NUMBA_NUM_THREADS":"1","OMP_NUM_THREADS":"1",
            "OPENBLAS_NUM_THREADS":"1","MKL_NUM_THREADS":"1"})
print("Running three direct-waveform GAs | maps",N_DISORDERS,"| seeds",SEEDS,flush=True)
subprocess.run(cmd,cwd=str(LIB),env=env,check=True)
print("Measured physical search complete.")

## 5. Reopen the saved physical-evaluation histories and audit the budget

The same tabular schema is presented for all representations; serialized control descriptors are loaded without fabricating values. The historical Legendre Try statistics use one row per attempted physical query per method. The report distinguishes method-history rows from distinct simulator calls when modal initial histories are reused.

In [ ]:
df=load_histories(HISTORY_DIR,REPRESENTATION)
assert set(df["map"])==set(MAPS),(set(df["map"]),set(MAPS))
assert set(df.seed)==set(SEEDS)
assert set(df.method)==set(METHODS)
from advanced_analysis import audit_budget
run_audit=audit_budget(df,MAPS,SEEDS,INITIAL_CONTROLS,GENERATIONS,TABLE_DIR,FIG_DIR,
                        shared_initial_cache=True)
display(run_audit.head(20))
df.drop(columns=["coefficients"],errors="ignore").to_csv(TABLE_DIR/"all_evaluations_without_vectors.csv",index=False)
print("Method-history observations",len(df),"valid",int(df.valid.sum()))

## 6. Optimization convergence and normalized improvement

Compare best actual physical reward versus attempted evaluations. Normalize against the best valid initial control for each map and seed (pooled across algorithms); do not confuse this with a zero-field physical baseline.

In [ ]:
initial_baselines=shared_initial_baselines(df)
plot_convergence(df,FIG_DIR,initial_baselines)

## 7. Exact evaluation to 95% and 99% of observed improvement

Use the first actual evaluation index at which each method attains the same map-level observed target. Missing targets are recorded as unreached, not extrapolated.

In [ ]:
targets=exact_target_attainment(df,TABLE_DIR)
display(targets.groupby(["method","fraction"],as_index=False).agg(reached=("reached","sum"),runs=("reached","count"),median_evaluations=("first_evaluation","median")))

## 8. Reference-map reward–complexity Pareto fronts

Complexity is the number of prominent extrema of the *rendered* driving field (2% peak-to-peak prominence). Reward is the measured terminal absolute curl.

In [ ]:
plot_reference_pareto(df,FIG_DIR)

## 9. Evolution of the physical waveform genotype

The first panel displays changes in the 900-sample waveform relative to the initial incumbent; the second displays the actual applied waveform. Each row corresponds to one of the three GA variants. The plot uses a single seed rather than pooling independent trajectories.

In [ ]:
plot_control_evolution(df,FIG_DIR,REPRESENTATION,seed=SEEDS[0])

## 10. Champion physical waveform and parsimonious alternative

Report the best measured control and, when available, the highest-reward solution with at most one prominent waveform turn. Descriptor plots remain representation-specific.

In [ ]:
plot_champion_and_descriptor(df,FIG_DIR,REPRESENTATION)

## 10b. Representation-specific descriptor information

There are no modal coefficients or a separate grammar: the genotype is the physical 900-sample waveform itself. The common modal-energy figure is not applicable, and no jump-count or jump-magnitude complexity descriptors are added.

In [ ]:
plot_modal_energy(df,FIG_DIR,REPRESENTATION)

## 11. Disorder-dependent winners and complexity

Best observed reward and complexity across N disorder maps. For a single map, omit meaningless between-map figures while retaining summary tables.

In [ ]:
map_stats=plot_disorder_summary(df,FIG_DIR,TABLE_DIR,MAPS)
display(map_stats.head(20))

## 12. Best and simple physical waveforms for each disorder

Four algorithms per map. Save the physical field shapes and exact evaluation indices that produced the measured rewards.

In [ ]:
plot_best_waveforms_by_disorder(df,MAPS,FIG_DIR,TABLE_DIR,all_maps=ANALYZE_EACH_DISORDER)

## 13. Measured Pareto fronts for every disorder

Create one four-method Pareto comparison per map, plus an all-map montage. These empirical frontiers depend only on controls actually evaluated.

In [ ]:
plot_pareto_by_disorder(df,MAPS,FIG_DIR,TABLE_DIR,all_maps=ANALYZE_EACH_DISORDER)
plot_all_disorder_pareto(df,MAPS,FIG_DIR)

## 14. Complexity retention and physical field descriptors

Compute 0/1/3-turn controls and the minimum complexity retaining 95/99% of each run's discovered gain. Also report total waveform variation, peak field, and the fraction of reward on the physical boundary.

In [ ]:
retention=complexity_retention(df,FIG_DIR,TABLE_DIR)
physical_metrics=plot_additional_complexity_metrics(df,FIG_DIR,TABLE_DIR)
display(retention.head(12));display(physical_metrics.head(12))

## 15. Disorder/seed variation, paired contrasts, provenance and surrogate-calibration status (not applicable to pure GA)

Replicate Legendre Try's advanced statistics for all five approaches. Paired comparison confidence intervals are bootstrapped over disorder maps, retaining optimizer seeds nested within each map. If there is only one independent disorder, the interval is undefined.

In [ ]:
winners,variance=seed_disorder_variance(df,FIG_DIR,TABLE_DIR)
paired=paired_method_comparison(winners,FIG_DIR,TABLE_DIR)
provenance=proposal_attribution(df,FIG_DIR,TABLE_DIR)
calibration=surrogate_calibration(df,FIG_DIR,TABLE_DIR)
display(variance);display(paired);display(calibration);display(provenance.head(20))

## 16. Cross-disorder transfer (optional; outside the optimizer budget)

Reevaluate the globally best and simplest controls from each source disorder on all target maps, without reoptimization. Save validity flags, raw measured curl, and normalized gain relative to each target's best valid initialization.

In [ ]:
transfer=[]
if RUN_CROSS_DISORDER_TRANSFER:
    for mid,g in df[df.valid&np.isfinite(df.reward)].groupby("map"):
        best=g.loc[g.reward.idxmax()]
        simple=g[g.complexity<=1]
        picks=[("best",best)]
        if len(simple):picks.append(("simple",simple.loc[simple.reward.idxmax()]))
        for kind,row in picks:
            w=np.asarray(fields(row.coefficients,n=900),float).reshape(-1)
            for target in MAPS:
                v=evaluate_fields(w,defects_ec=defect_for(target),method="rk4",
                                  fidelity=1,max_substeps=100000)
                good=bool(v["valid"][0]) and not bool(v["native_guard_exceeded"][0])
                y=float(v["reward"][0]) if good else np.nan
                initial=df[(df["map"]==target)&(df.generation==0)&df.valid].reward
                optimum=df[(df["map"]==target)&df.valid].reward
                base=float(initial.max()) if len(initial) else np.nan
                best_target=float(optimum.max()) if len(optimum) else np.nan
                relative=(y-base)/(best_target-base) if good and np.isfinite(base) and best_target>base+1e-12 else np.nan
                transfer.append(dict(source=mid,target=target,selection=kind,reward=y,valid=good,normalized_gain=relative))
    transfer_df=pd.DataFrame(transfer)
    transfer_df.to_csv(TABLE_DIR/"transfer.csv",index=False)
    display(transfer_df.head(12))
    if len(MAPS)>1 and not transfer_df.empty:
        for metric in ["reward","normalized_gain"]:
            fig,axs=plt.subplots(1,len(transfer_df.selection.unique()),figsize=(6*len(transfer_df.selection.unique()),5),squeeze=False,layout="constrained")
            for ax,selection in zip(axs.flat,transfer_df.selection.unique()):
                d=transfer_df[(transfer_df.selection==selection)&transfer_df.valid]
                mat=d.pivot(index="source",columns="target",values=metric).reindex(index=MAPS,columns=MAPS)
                image=ax.imshow(mat,cmap="viridis",aspect="auto")
                ax.set(title=selection+" | "+metric,xticks=range(len(MAPS)),xticklabels=MAPS,yticks=range(len(MAPS)),yticklabels=MAPS)
                ax.tick_params(axis="x",rotation=90);fig.colorbar(image,ax=ax)
            fig.savefig(FIG_DIR/("07_transfer_"+metric+".png"),dpi=145)
            plt.show();plt.close(fig)
else:print("Transfer disabled; no additional cross-disorder physical calls.")

## 17. Higher-fidelity RK4 and terminal spatial validation (optional)

Independently reevaluate the selected champion(s) at RK4 fidelity 2. Save terminal physical maps and whole/interior/boundary curl; this only refines integration, not the time grid of the imposed waveform.

In [ ]:
checks=[]
if RUN_HIGHER_FIDELITY_VALIDATION:
    selected_maps=MAPS if VALIDATE_ALL_DISORDERS else ["reference"]
    for mid in selected_maps:
        subset=df[(df["map"]==mid)&df.valid&np.isfinite(df.reward)]
        if subset.empty:continue
        champion=subset.loc[subset.reward.idxmax()]
        simple=subset[subset.complexity<=1]
        controls=[("best",champion)]
        if len(simple):
            s=simple.loc[simple.reward.idxmax()]
            if s.name!=champion.name:controls.append(("simple",s))
        fig,axs=plt.subplots(len(controls),3,figsize=(11,3.6*len(controls)),squeeze=False,layout="constrained")
        for i,(name,row) in enumerate(controls):
            w=np.asarray(fields(row.coefficients,n=900),float).reshape(-1)
            r=evaluate_fields(w,defects_ec=defect_for(mid),method="rk4",fidelity=2,max_substeps=100000)
            ok=bool(r["valid"][0]) and not bool(r["native_guard_exceeded"][0])
            if ok:
                P=r["final_P"][0];curl=np.abs(curl_field(P))
                for j,(array,title) in enumerate(zip((P[0],P[1],curl),("Px","Py","|curl|"))):
                    ax=axs[i,j];im=ax.imshow(array,origin="lower");ax.set_title(f"{mid} {name}: {title}");fig.colorbar(im,ax=ax)
            else:
                for ax in axs[i]:ax.text(.5,.5,"Invalid refined trajectory",ha="center",va="center",transform=ax.transAxes)
            y=float(r["reward"][0]) if ok else np.nan
            interior=float(r["interior"][0]) if ok else np.nan
            checks.append(dict(map=mid,selection=name,original_reward=float(row.reward),
               refined_reward=y,difference=y-float(row.reward) if ok else np.nan,
               interior_reward=interior,boundary_reward=y-interior if ok else np.nan,valid=ok))
        fig.savefig(FIG_DIR/f"08_spatial_validation_{mid}.png",dpi=150)
        plt.show();plt.close(fig)
    pd.DataFrame(checks).to_csv(TABLE_DIR/"higher_fidelity_validation.csv",index=False)
    display(pd.DataFrame(checks).head(20))
else:print("Independent numerical refinement disabled.")

## 18. Physical texture evolution as optimizer reward increases — NEW IN v4

**No additional reward function.** Save the final (P_x,P_y) and curl for every distinct best-so-far waveform, with an index for all generations. The code independently repeats the selected FerroSim calculation and rejects an archived winner if its replay reward disagrees. Compute curl localization, high-curl area relative to an initialization-defined common threshold, successive polarization-state change, and spatial correlation length. Output T1 snapshots, T2 trajectory metrics, T3 reward–texture relationships, and T4 PCA of the discovered polarization states. This is **optimization time**, not simulation time within one waveform.

In [ ]:
if RUN_TEXTURE_ANALYSIS:
    TEXTURE_DIR=RUN_ROOT/"texture"
    texture_timeline,texture_states=analyze_textures(
        df,MAPS,MAP_DIR,TEXTURE_DIR,max_substeps=100000,
        max_snapshots=TEXTURE_MAX_SNAPSHOTS,
        plot_each_disorder=TEXTURE_PLOT_EACH_DISORDER,
        plot_pca_reference_only=TEXTURE_PCA_REFERENCE_ONLY)
    display(texture_states.head(16))
    display(texture_timeline.head(16))
    print("Saved physical configuration archive:",TEXTURE_DIR.resolve())
else:print("Texture analysis disabled (no physical-state archive generated).")

## 19. Archive source hashes, configurations, all results, and output index

The notebook retains physical histories, restartable checkpoints, per-disorder figures, CSV/NPZ records and texture archives in a source+configuration-fingerprinted directory. Original the five other v4 notebooks are not overwritten.

In [ ]:
from common_analysis import write_selected
write_selected(df,TABLE_DIR)
(RUN_ROOT/"config.json").write_text(json.dumps(SEARCH_CONFIG | {
    "hyperparameters":hyper,"maps":MAPS,
    "analysis_flags":{"all_disorders":ANALYZE_EACH_DISORDER,
       "transfer":RUN_CROSS_DISORDER_TRANSFER,
       "numerical_refinement":RUN_HIGHER_FIDELITY_VALIDATION,
       "texture_analysis":RUN_TEXTURE_ANALYSIS}},indent=2))
export_index(FIG_DIR,TABLE_DIR,RUN_ROOT)
print("Completed",REPRESENTATION,"v4; run directory:",RUN_ROOT)
print("Saved figures",len(list(FIG_DIR.rglob("*.png"))),
      "tables",len(list(TABLE_DIR.rglob("*.csv"))))